# Notebook 12 — Multinomial Logistic Regression: Cluster Membership from Triage Variables

# ==============================================================================
# CHUNK 0 — Notebook overview and analytical strategy
# ==============================================================================

## Purpose

This notebook quantifies the association between **variables available at triage** (before any
resource consumption is recorded) and **resource utilisation cluster membership** derived from
the HDBSCAN clustering analysis. The goal is not prediction but description: to characterise
which patient profiles at admission are independently associated with each consumption pattern.

---

## Analytical strategy

### Why logistic regression?
Cluster labels are a nominal outcome. Multinomial logistic regression (MNLogit) models the
log-odds of belonging to each cluster relative to a clinical reference group, yielding
**odds ratios (OR)** with 95% confidence intervals — interpretable as the relative likelihood
of a patient with a given characteristic being in cluster Ck rather than the reference cluster.

### Reference cluster
The reference cluster in each solution is the **largest, clinically least-intensive group**:
- 5-cluster solution: C5 (Discharged + minimal consumption)
- 9-cluster solution: C9 (Discharged + minimal consumption)

This choice maximises statistical stability (large reference group) and clinical
interpretability (OR > 1 = more resource-intensive than a "simple" visit).

### Two models per solution
Each analysis is run twice:
- **Model 1 — With triage:** includes triage level as a covariate
- **Model 2 — Without triage:** excludes triage level

The comparison of these two models (McFadden R², LRT) quantifies the **independent
contribution of triage** to cluster prediction, and tests whether consumption clusters
capture information beyond what triage alone encodes.

### On statistical significance
With N ≈ 50,000, virtually all associations will reach statistical significance (p < 0.05).
P-values are reported but **clinical relevance is assessed by OR magnitude**:
- OR > 1.5 or OR < 0.67 = clinically meaningful threshold (50% increase or decrease in odds)

---

## Notebook structure

| Chunk | Content |
|-------|---------|
| 1 | Imports, constants, variable mappings, feature lists |
| 1bis | Data loading, remaps, cell count check |
| 2 | Reference categories + `format_mnlogit_results()` helper |
| 3 | Spline analysis of age (justifies categorical age_group) |
| 3bis | Binary splines — cluster k vs rest |
| 4 | **Univariate** MNLogit — one variable at a time, all clusters vs reference |
| 4b | **Univariate binary** logistic regression — Outliers vs rest |
| 5 | **Multivariate** MNLogit (standard) — with and without triage |
| 5bis | **Ridge L2 multivariate** — grouped vital signs (not_measured/normal/abnormal) |
| 5ter | **Ridge L2 multivariate** — detailed vital signs (original modalities) |
| 5quater | **Binary logistic regression** — Outliers vs all clusters (multivariate) |

---

## On quasi-separation and Ridge regularization

Standard MNLogit (chunks 4–5) encounters **perfect quasi-separation** for clusters C2 and C4
in the 9-cluster solution: certain combinations of complaint category and vital sign status
perfectly predict membership in these clusters, making likelihood maximization impossible.

This is **not a methodological failure** — it is a substantive finding reflecting the
exceptional clinical homogeneity of these clusters. Their patients follow highly stereotyped
care pathways that are entirely determined by a small number of triage variables.

**Ridge L2 regularization** (chunks 5bis–5ter) addresses this by penalizing large
coefficients, yielding finite and comparable estimates for all clusters simultaneously.
The regularization parameter λ (equivalently C = 1/λ in sklearn) is selected by
5-fold cross-validation. Confidence intervals are derived by bootstrap resampling (B = 200).

Note that Ridge OR are slightly attenuated towards 1.0 compared to unpenalised MLE — this
is expected and acceptable for a descriptive analysis. Both sets of results (standard MNLogit
and Ridge) are presented for transparency.

---

## Variable encoding

All categorical variables are one-hot encoded with explicit reference categories:

| Variable | Reference |
|----------|-----------|
| sex | M (male) |
| age_group | 15–30 |
| complaint_category_reg | Trauma |
| triage_grouped | 3 (semi-urgent) |
| transport_grouped | Personal |
| Vital signs (chunks 4–5bis) | normal / not_measured depending on variable |
| Vital signs (chunk 5ter) | original reference modality (normotension, normocardia, etc.) |
"""

In [294]:
# ==============================================================================
# CHUNK 1 — Imports, constants, mappings
# ==============================================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcdefaults()
plt.style.use('default')
plt.rcParams['text.color']        = 'black'
plt.rcParams['axes.labelcolor']   = 'black'
plt.rcParams['xtick.color']       = 'black'
plt.rcParams['ytick.color']       = 'black'
plt.rcParams['figure.facecolor']  = 'white'
plt.rcParams['axes.facecolor']    = 'white'
plt.rcParams['savefig.facecolor'] = 'white'

import seaborn as sns
import statsmodels.api as sm
from statsmodels.formula.api import mnlogit
from patsy import dmatrix, cr
import warnings
warnings.filterwarnings("ignore")
import re
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder


# ── Paths ──────────────────────────────────────────────────────────────────────
RUN_LABEL = "s2_balanced"
SCALER    = "minmax"
BASE_DIR  = f"Results/Regular_clustering/Full_dataset/With_counts/{SCALER}/{RUN_LABEL}"

# ── Cluster labels mapping ─────────────────────────────────────────────────────
CLUSTERING_RUNS = [
    {
        'n_clusters'    : 5,
        'mcs'           : 3407,
        'ms'            : 170,
        'ref_cluster'   : 3,
        'cluster_labels': {
            1 : "C1 — UHCD + hospitalization + heavy workup",
            0 : "C2 — Hospitalized + full workup",
            2 : "C3 — Discharged + biology +/- ECG",
            4 : "C4 — Discharged + isolated X-ray +/- CT",
            3 : "C5 — Discharged + minimal consumption",
            -1: "Outliers",
        },
        'cluster_order' : [
            "C1 — UHCD + hospitalization + heavy workup",
            "C2 — Hospitalized + full workup",
            "C3 — Discharged + biology +/- ECG",
            "C4 — Discharged + isolated X-ray +/- CT",
            "C5 — Discharged + minimal consumption",
            "Outliers",
        ],
    },
    {
        'n_clusters'    : 9,
        'mcs'           : 2271,
        'ms'            : 15,
        'ref_cluster'   : 2,
        'cluster_labels': {
            0 : "C1 — UHCD + hospitalization + mixed workup ++",
            8 : "C2 — Hospitalized + bio +++ (bloodtest++, culture+++, bloodgas+) + imaging+(ctscan) + ECG+",
            5 : "C3 — Hospitalized + blood test++ CTScan++ ECG+",
            6 : "C4 — Hospitalized + blood test++ MRI+++ ECG++",
            4 : "C5 — Hospitalized + blood test + ECG +/- X-ray ++",
            7 : "C6 — Hospitalized + isolated imaging (xray,ctscan)",
            1 : "C7 — Discharged + biology + ECG+",
            3 : "C8 — Discharged + isolated X-ray",
            2 : "C9 — Discharged + minimal consumption",
            -1: "Outliers",
        },
        'cluster_order' : [
            "C1 — UHCD + hospitalization + mixed workup ++",
            "C2 — Hospitalized + bio +++ (bloodtest++, culture+++, bloodgas+) + imaging+(ctscan) + ECG+",
            "C3 — Hospitalized + blood test++ CTScan++ ECG+",
            "C4 — Hospitalized + blood test++ MRI+++ ECG++",
            "C5 — Hospitalized + blood test + ECG +/- X-ray ++",
            "C6 — Hospitalized + isolated imaging (xray,ctscan)",
            "C7 — Discharged + biology + ECG+",
            "C8 — Discharged + isolated X-ray",
            "C9 — Discharged + minimal consumption",
            "Outliers",
        ],
    },
]

# ── Rare complaint categories to merge ────────────────────────────────────────
SYSTEMIC_RARE = [
    "Metabolic_Endocrine",
    "Hematology",
    "Poisoning_Intoxication",
    "Infectious_Fever",
    "Respiratory",
    "Urogenital_Renal",
    "Administrative_Social_Unclassified",
]

# ── Vital sign remap : not_measured / normal / abnormal ───────────────────────
# VITAL_SIGN_REMAP = {
#     'bp_status': {
#         'not_measured' : 'not_measured',
#         'hypotension'  : 'abnormal',
#         'normotension' : 'normal',
#         'hypertension' : 'abnormal',
#     },
#     'hr_status': {
#         'not_measured' : 'not_measured',
#         'bradycardia'  : 'abnormal',
#         'normocardia'  : 'normal',
#         'tachycardia'  : 'abnormal',
#     },
#     'temp_status': {
#         'not_measured' : 'not_measured',
#         'hypothermia'  : 'abnormal',
#         'normothermia' : 'normal',
#         'hyperthermia' : 'abnormal',
#     },
#     'sat_status': {
#         'not_measured'  : 'not_measured',
#         'severe hypoxia': 'abnormal',
#         'hypoxia'       : 'abnormal',
#         'normal'        : 'normal',
#     },
#     'rr_status': {
#         'not_measured' : 'not_measured',
#         'bradypnea'    : 'abnormal',
#         'normal'       : 'normal',
#         'tachypnea'    : 'abnormal',
#     },
#     'o2_flow_status': {
#         'not_measured' : 'not_measured',
#         'off'          : 'normal',
#         'on'           : 'abnormal',
#     },
#     'gcs_status': {
#         'not_measured'       : 'not_measured',
#         'severe_impairment'  : 'abnormal',
#         'moderate_impairment': 'abnormal',
#         'normal'             : 'normal',
#     },
#     'cap_blood_sugar_status': {
#         'not_measured' : 'not_measured',
#         'hypoglycemia' : 'abnormal',
#         'normoglycemia': 'normal',
#         'hyperglycemia': 'abnormal',
#     },
#     'pupils_status': {
#         'not_measured' : 'not_measured',
#         'myosis'       : 'abnormal',
#         'normal'       : 'normal',
#         'mydriasis'    : 'abnormal',
#     },
#     'anisocoria_status': {
#         'not_measured' : 'not_measured',
#         'no'           : 'normal',
#         'yes'          : 'abnormal',
#     },
#     'urine_dipstick_clean_status': {
#         'not_measured' : 'not_measured',
#         'negative'     : 'normal',
#         'positive'     : 'abnormal',
#     },
#     'pain_status': {
#         'not_measured' : 'not_measured',
#         'no_pain'      : 'normal',
#         'mild_pain'    : 'abnormal',
#         'moderate_pain': 'abnormal',
#         'severe_pain'  : 'abnormal',
#     },
#     'breathalyzer_status': {
#         'not_measured' : 'not_measured',
#         'negative'     : 'normal',
#         'positive'     : 'abnormal',
#     },
#     'hemocue_status': {
#         'not_measured'   : 'not_measured',
#         'severe_anemia'  : 'abnormal',
#         'moderate_anemia': 'abnormal',
#         'normal'         : 'normal',
#     },
# }

# ── Triage grouping ────────────────────────────────────────────────────────────
TRIAGE_GROUP_MAP = {1: "1-2", 2: "1-2", 3: "3", 4: "4-5", 5: "4-5"}

# ── Feature lists ──────────────────────────────────────────────────────────────
REG_FEATURES_STATUS = [
    "bp_status", "hr_status", "temp_status", "sat_status",
    "rr_status", "o2_flow_status", "gcs_status",
    "cap_blood_sugar_status", "anisocoria_status",
    "urine_dipstick_clean_status", "pain_status",
    "breathalyzer_status", "hemocue_status",
]

EXCLUDED_VARS = [
    'breathalyzer_status',
    'hemocue_status',
    'anisocoria_status',
]

REG_FEATURES_STATUS_CLEAN = [v for v in REG_FEATURES_STATUS if v not in EXCLUDED_VARS]

REG_FEATURES_CATEG_WITH_TRIAGE = [
    "sex", "transport_grouped", "age_group",
    "complaint_category_reg", "triage_grouped"
]

REG_FEATURES_CATEG_NO_TRIAGE = [
    "sex", "transport_grouped", "age_group",
    "complaint_category_reg"
]

ALL_REG_FEATURES_WITH_TRIAGE = REG_FEATURES_CATEG_WITH_TRIAGE + REG_FEATURES_STATUS#_CLEAN
ALL_REG_FEATURES_NO_TRIAGE   = REG_FEATURES_CATEG_NO_TRIAGE   + REG_FEATURES_STATUS#_CLEAN

print("Chunk 1 loaded.")

# ── Variables excluded for multivariate (quasi-separation) ────────────────────
EXCLUDED_VARS_MULTI = EXCLUDED_VARS + [
    'urine_dipstick_clean_status',
    'cap_blood_sugar_status',
    'o2_flow_status',
]

REG_FEATURES_STATUS_MULTI = [v for v in REG_FEATURES_STATUS if v not in EXCLUDED_VARS_MULTI]

ALL_REG_FEATURES_WITH_TRIAGE_MULTI = REG_FEATURES_CATEG_WITH_TRIAGE + REG_FEATURES_STATUS_MULTI
ALL_REG_FEATURES_NO_TRIAGE_MULTI   = REG_FEATURES_CATEG_NO_TRIAGE   + REG_FEATURES_STATUS_MULTI

Chunk 1 loaded.


In [295]:
# ==============================================================================
# CHUNK 1bis — Data loading + remaps + cell count check
# ==============================================================================

# ── Load ───────────────────────────────────────────────────────────────────────
config  = CLUSTERING_RUNS[1]   # index 1 = 9 clusters, index 0 = 5 clusters
mcs, ms = config['mcs'], config['ms']
CSV_PATH = f"{BASE_DIR}/final_mcs{mcs}_ms{ms}/clustering_{RUN_LABEL}_{SCALER}_mcs{mcs}_ms{ms}_labeled.csv"

df_clust = pd.read_csv(CSV_PATH, low_memory=False)

# ── Apply vital sign remap ─────────────────────────────────────────────────────
for col, mapping in VITAL_SIGN_REMAP.items():
    if col in df_clust.columns:
        df_clust[col] = df_clust[col].map(mapping)

# ── Triage grouped ─────────────────────────────────────────────────────────────
df_clust['triage_grouped'] = (
    df_clust['triage'].astype(float).astype(int)
    .map(TRIAGE_GROUP_MAP).astype(str)
)

# ── Complaint category grouped ─────────────────────────────────────────────────
df_clust['complaint_category_reg'] = df_clust['complaint_category'].replace(
    {cat: 'Other' for cat in SYSTEMIC_RARE}
)

# ── Build df_reg ───────────────────────────────────────────────────────────────
cols_needed = list(set(
    ALL_REG_FEATURES_WITH_TRIAGE + ALL_REG_FEATURES_NO_TRIAGE +
    ['cluster', 'cluster_label', 'age', 'triage', 'complaint_category']
))
df_reg = df_clust[[c for c in cols_needed if c in df_clust.columns]].copy()
df_reg = df_reg.dropna(subset=ALL_REG_FEATURES_WITH_TRIAGE + ['cluster'])
df_reg['cluster'] = df_reg['cluster'].astype(int)

# ── Cell count check ───────────────────────────────────────────────────────────
print("=" * 70)
print("CELL COUNT CHECK — quasi-separation detection")
print("=" * 70)

for var in ALL_REG_FEATURES_WITH_TRIAGE + ALL_REG_FEATURES_NO_TRIAGE:
    if var not in df_reg.columns:
        print(f"MISSING: {var}")
        continue
    ct = pd.crosstab(df_reg[var], df_reg['cluster'])
    min_cell = ct.min().min()
    flag = "⚠️  QUASI-SEP RISK" if min_cell < 10 else "✅"
    print(f"\n{flag}  {var} — min cell = {min_cell}")
    if min_cell < 10:
        print(ct)

CELL COUNT CHECK — quasi-separation detection

✅  sex — min cell = 1024

✅  transport_grouped — min cell = 10

✅  age_group — min cell = 227

⚠️  QUASI-SEP RISK  complaint_category_reg — min cell = 2
cluster                     -1     0     1     2     3     4     5     6  \
complaint_category_reg                                                     
Abdominal_Digestive        759  1043   803  1388    58   775  1120     4   
Cardiovascular             157   397   373   412   207   941   193    30   
Dermatological             116   116   195  1025    35    74   182     2   
ENT_Ophthalmology_Dental    95    52   103  1356     8    69    70     9   
General_Deterioration      163  1153   153   185    42   172   244    62   
Medical_Followup           160   384   160   447    60   142   117    23   
Musculoskeletal            645   586   440  2001   535   394   317    88   
Neurological               837  3075  1158  1356    40  1340  1654  2197   
Other                     1037  1865   4

Le cluster de référence doit être le plus grand ou le plus "basal" cliniquement — par exemple le cluster des patients jeunes, non urgents, faible consommation. Tous les OR s'interprètent par rapport à lui.
Avec N=120 000 tout sera significatif comme pour Cramér's V — donc ici aussi tu regardes la magnitude de l'OR, pas juste le p-value. Un OR de 1.05 n'est pas intéressant même si p < 0.001. Concentre-toi sur les OR > 1.5 ou < 0.67 (effet modéré).

In [296]:
# # ── Variables excluded due to quasi-separation ────────────────────────────────
# EXCLUDED_VARS = [
#     'breathalyzer_status',
#     'hemocue_status',
#     'anisocoria_status',
# ]
#
# REG_FEATURES_STATUS_CLEAN = [v for v in REG_FEATURES_STATUS if v not in EXCLUDED_VARS]
#
# # ── Feature lists ──────────────────────────────────────────────────────────────
# REG_FEATURES_CATEG_WITH_TRIAGE = [
#     "sex", "transport_grouped", "age_group",
#     "complaint_category_reg", "triage_grouped"
# ]
#
# REG_FEATURES_CATEG_NO_TRIAGE = [
#     "sex", "transport_grouped", "age_group",
#     "complaint_category_reg"
# ]
#
# ALL_REG_FEATURES_WITH_TRIAGE = REG_FEATURES_CATEG_WITH_TRIAGE + REG_FEATURES_STATUS_CLEAN
# ALL_REG_FEATURES_NO_TRIAGE   = REG_FEATURES_CATEG_NO_TRIAGE   + REG_FEATURES_STATUS_CLEAN

In [297]:
# ==============================================================================
# CHUNK 2 — Reference categories + helper function
# ==============================================================================

# ── Reference categories ───────────────────────────────────────────────────────
REF_CATEGORIES = {
    "sex"                         : "M",
    "transport_grouped"           : "Personal",
    "age_group"                   : "15-30",
    "complaint_category_reg"      : "Trauma",
    "triage_grouped"              : "3",        # reference = triage 3 (semi-urgent)
    "bp_status"                   : "normal",
    "hr_status"                   : "normal",
    "temp_status"                 : "normal",
    "sat_status"                  : "normal",
    "rr_status"                   : "normal",
    "o2_flow_status"              : "normal",
    "gcs_status"                  : "normal",
    "cap_blood_sugar_status"      : "normal",
    "urine_dipstick_clean_status" : "normal",
    "pain_status"                 : "normal",
}

# ── Helper — format results table ─────────────────────────────────────────────
def format_mnlogit_results(result, ref_cluster, CLUSTER_LABELS):
    rows = []
    conf = result.conf_int()

    param_cols_no_ref = [c for c in result.params.columns if c != ref_cluster]
    conf_keys         = conf.index.get_level_values(0).unique().tolist()

    for col_idx, cluster in enumerate(param_cols_no_ref):
        params  = result.params[cluster]
        pvalues = result.pvalues[cluster]

        try:
            conf_clust = conf.xs(cluster)
        except KeyError:
            conf_clust = conf.xs(conf_keys[col_idx])

        # ── DIAGNOSTIC ────────────────────────────────────────────────────────
        if col_idx == 0:
            print("\nconf_clust columns:", conf_clust.columns.tolist())
            print("conf_clust head:\n", conf_clust.head(3))
            print("\nparams head:\n", params.head(3))
            var_test = params.index[1]  # première variable non-Intercept
            or_manual = round(np.exp(params[var_test]), 3)
            ci_row = conf_clust.loc[var_test]
            ci_low_manual  = round(np.exp(ci_row.iloc[0]), 3)
            ci_high_manual = round(np.exp(ci_row.iloc[1]), 3)
            print(f"\nManual check — {var_test}")
            print(f"  OR     : {or_manual}")
            print(f"  CI_low : {ci_low_manual}")
            print(f"  CI_high: {ci_high_manual}")
            print(f"  Valid  : {ci_low_manual <= or_manual <= ci_high_manual}")
        # ─────────────────────────────────────────────────────────────────────

        for var in params.index:
            if var == "Intercept":
                continue

            or_val = np.exp(params[var])
            pval   = pvalues[var]

            try:
                row     = conf_clust.loc[var]
                ci_low  = np.exp(row["lower"])
                ci_high = np.exp(row["upper"])
            except KeyError:
                idx     = list(params.index).index(var)
                ci_low  = np.exp(conf_clust.iloc[idx]["lower"])
                ci_high = np.exp(conf_clust.iloc[idx]["upper"])

            clean_var = re.sub(
                r"C\((.+?),\s*Treatment\(.+?\)\)\[T\.(.+?)\]",
                r"\1 = \2", var
            )

            rows.append({
                "cluster_vs_ref": f"{CLUSTER_LABELS.get(cluster, f'C{cluster}')} vs {CLUSTER_LABELS.get(ref_cluster, f'C{ref_cluster}')}",
                "variable"      : clean_var,
                "OR"            : round(or_val,  3),
                "CI_low"        : round(ci_low,  3),
                "CI_high"       : round(ci_high, 3),
                "p_value"       : round(pval,    4),
                "significant"   : pval < 0.05,
            })

    return pd.DataFrame(rows)


print("Reference categories set:")
for k, v in REF_CATEGORIES.items():
    print(f"  {k:<35} ref = {v}")
print("\nHelper function loaded.")

print(f"Clusters dans df_multivariate : {sorted(df_multivariate['cluster_vs_ref'].unique())}")
print(f"Colonnes params du modèle : {model_multi.params.columns.tolist()}")
print(f"ref_c : {ref_c}")

Reference categories set:
  sex                                 ref = M
  transport_grouped                   ref = Personal
  age_group                           ref = 15-30
  complaint_category_reg              ref = Trauma
  triage_grouped                      ref = 3
  bp_status                           ref = normal
  hr_status                           ref = normal
  temp_status                         ref = normal
  sat_status                          ref = normal
  rr_status                           ref = normal
  o2_flow_status                      ref = normal
  gcs_status                          ref = normal
  cap_blood_sugar_status              ref = normal
  urine_dipstick_clean_status         ref = normal
  pain_status                         ref = normal

Helper function loaded.
Clusters dans df_multivariate : ['C1 — UHCD + hospitalization + mixed workup ++ vs C9 — Discharged + minimal consumption', 'C3 — Hospitalized + blood test++ CTScan++ ECG+ vs C9 — Discharged + m

In [298]:
# # ==============================================================================
# # CHUNK 3 — Splines — age vs cluster membership (multinomial vs ref clsuter)
# # ==============================================================================
#
# for config in CLUSTERING_RUNS:
#     n_clusters     = config['n_clusters']
#     mcs            = config['mcs']
#     ms             = config['ms']
#     ref_c          = config['ref_cluster']
#     CLUSTER_LABELS = config['cluster_labels']
#     CLUSTER_ORDER  = config['cluster_order']
#
#     # ── Paths ─────────────────────────────────────────────────────────────────
#     CSV_PATH    = f"{BASE_DIR}/final_mcs{mcs}_ms{ms}/clustering_{RUN_LABEL}_{SCALER}_mcs{mcs}_ms{ms}_labeled.csv"
#     OUT_DIR_REG = f"{BASE_DIR}/final_mcs{mcs}_ms{ms}/logistic_regression"
#     os.makedirs(OUT_DIR_REG, exist_ok=True)
#
#     # ── Load ──────────────────────────────────────────────────────────────────
#     df_clust = pd.read_csv(CSV_PATH, low_memory=False)
#     df_clust['complaint_category_reg'] = df_clust['complaint_category'].replace(
#         {cat: 'Metabolic_Hematologic_Toxic' for cat in SYSTEMIC_RARE}
#     )
#     df_clust['cluster_label'] = df_clust['cluster'].map(CLUSTER_LABELS)
#     df_clust['cluster_label'] = pd.Categorical(
#         df_clust['cluster_label'], categories=CLUSTER_ORDER, ordered=True
#     )
#
#     # ── Build df_reg ──────────────────────────────────────────────────────────
#     cols_needed = ALL_REG_FEATURES_WITH_TRIAGE + ['cluster', 'cluster_label', 'age']
#     df_reg = df_clust[[c for c in cols_needed if c in df_clust.columns]].copy()
#     df_reg = df_reg.dropna(subset=ALL_REG_FEATURES_WITH_TRIAGE + ['cluster'])
#
#     df_reg['cluster']                = df_reg['cluster'].astype(int)
#     df_reg['triage']                 = df_reg['triage'].astype(float).astype(int).astype(str)
#     df_reg['age_group']              = df_reg['age_group'].astype(str)
#     df_reg['sex']                    = df_reg['sex'].astype(str)
#     df_reg['transport_grouped']      = df_reg['transport_grouped'].astype(str)
#     df_reg['complaint_category_reg'] = df_reg['complaint_category_reg'].astype(str)
#     for col in REG_FEATURES_STATUS:
#         if col in df_reg.columns:
#             df_reg[col] = df_reg[col].astype(str)
#
#     df_reg['is_outlier'] = (df_reg['cluster'] == -1).astype(int)
#
#     print(f"\n{'='*60}")
#     print(f"RUN — {n_clusters} clusters (mcs={mcs}, ms={ms})")
#     print(f"{'='*60}")
#     print(f"Regression dataset: {len(df_reg):,} patients")
#
#
#
#
#     OUT_DIR_SPLINE = os.path.join(OUT_DIR_REG, "splines")
#     ref_c_spline = ref_c
#     ref_label    = CLUSTER_LABELS[ref_c_spline]
#     os.makedirs(OUT_DIR_SPLINE, exist_ok=True)
#
#
#
#     df_spline = df_reg[["age", "cluster"]].dropna()
#     df_spline = df_spline[df_spline["age"].between(14, 120)].copy()
#
#     # ── Cluster 3 (C5) = référence → en premier dans l'encodage ──────────────────
#     df_spline["cluster"] = pd.Categorical(
#         df_spline["cluster"],
#         categories=[ref_c_spline] + sorted([c for c in df_spline["cluster"].unique() if c != ref_c_spline])
#     )
#
#     age_range = np.linspace(df_spline["age"].min(), df_spline["age"].max(), 300)
#
#     spline_basis       = dmatrix("cr(age, df=4) - 1", {"age": df_spline["age"]}, return_type="dataframe")
#     spline_basis_range = dmatrix("cr(age, df=4) - 1", {"age": age_range},        return_type="dataframe")
#
#     spline_basis_model = sm.add_constant(spline_basis)
#     spline_basis_pred  = sm.add_constant(spline_basis_range)
#
#     # ── MNLogit ───────────────────────────────────────────────────────────────────
#     y_cat            = df_spline["cluster"].cat.codes
#     model            = sm.MNLogit(y_cat, spline_basis_model).fit(method="newton", maxiter=500, disp=False)
#     # Trier non_ref_clusters selon CLUSTER_ORDER
#     non_ref_clusters = [
#         c for label in CLUSTER_ORDER
#         for c, lbl in CLUSTER_LABELS.items()
#         if lbl == label and c != ref_c_spline and c in df_spline["cluster"].unique()
#     ]
#     n_outcomes       = len(non_ref_clusters)
#     logit_preds      = spline_basis_pred.values @ model.params.values
#
#     # ── IC ────────────────────────────────────────────────────────────────────────
#     cov      = model.cov_params()
#     n_params = spline_basis_pred.shape[1]
#     ci_lows, ci_highs = [], []
#
#     for k in range(n_outcomes):
#         idx   = slice(k * n_params, (k + 1) * n_params)
#         cov_k = cov.values[idx, idx]
#         grad  = spline_basis_pred.values
#         se_k  = np.sqrt((grad @ cov_k @ grad.T).diagonal())
#         ci_lows.append(logit_preds[:, k] - 1.96 * se_k)
#         ci_highs.append(logit_preds[:, k] + 1.96 * se_k)
#
#     # ── FIGURE 1 — Un subplot par cluster non-référence ──────────────────────────
#     palette = sns.color_palette("tab10", n_outcomes)
#     n_cols  = min(3, n_outcomes)
#     n_rows  = (n_outcomes + n_cols - 1) // n_cols
#
#     fig, axes = plt.subplots(n_rows, n_cols,
#                               figsize=(n_cols * 6, n_rows * 5), facecolor="white")
#     fig.patch.set_facecolor("white")
#     axes = np.array(axes).flatten()
#
#     df_spline["age_bin"] = pd.cut(df_spline["age"], bins=20)
#
#     for i, (c, color) in enumerate(zip(non_ref_clusters, palette)):
#         ax        = axes[i]
#         c_label   = CLUSTER_LABELS.get(c, f"Cluster {c}")
#
#         # ── Points bruts cohérents avec MNLogit (c vs ref uniquement) ────────────
#         df_pair = df_spline[df_spline["cluster"].isin([c, ref_c_spline])]
#         raw = (
#             df_pair.groupby("age_bin", observed=True)
#             .apply(lambda x: (x["cluster"] == c).mean())
#             .reset_index()
#         )
#         raw["age_mid"]   = raw["age_bin"].apply(lambda x: x.mid)
#         raw["logit_raw"] = np.log(raw[0] / (1 - raw[0]))
#         raw = raw[(raw[0] > 0) & (raw[0] < 1)]
#
#         ax.scatter(raw["age_mid"], raw["logit_raw"],
#                    color="grey", s=20, alpha=0.5, label="Raw proportion (logit)")
#         ax.plot(age_range, logit_preds[:, i],
#                 color=color, linewidth=2, label="Spline fit")
#         ax.fill_between(age_range, ci_lows[i], ci_highs[i],
#                         alpha=0.2, color=color, label="95% CI")
#
#         for boundary in [30, 45, 60, 75]:
#             ax.axvline(boundary, color="lightgrey", linestyle="--", linewidth=0.8)
#
#         ax.set_facecolor("white")
#         ax.set_title(f"{c_label}\nvs {ref_label} (ref)",
#                      fontsize=10, fontweight="bold", color="black")
#         ax.set_xlabel("Age", fontsize=10, color="black")
#         ax.set_ylabel(f"Log-odds vs {ref_label}", fontsize=9, color="black")
#         ax.tick_params(colors="black")
#         ax.spines["top"].set_visible(False)
#         ax.spines["right"].set_visible(False)
#
#     for j in range(i + 1, len(axes)):
#         axes[j].set_visible(False)
#
#     legend_elements = [
#         Line2D([0], [0], color="grey",      linewidth=0, marker="o",
#                markersize=6, alpha=0.5, label="Raw proportion (logit)"),
#         Line2D([0], [0], color="black",     linewidth=2,  label="Spline fit"),
#         Line2D([0], [0], color="black",     linewidth=8,  alpha=0.2, label="95% CI"),
#         Line2D([0], [0], color="lightgrey", linewidth=1,  linestyle="--", label="Age group boundary"),
#     ]
#     fig.legend(handles=legend_elements, loc="lower center", ncol=4,
#                fontsize=9, frameon=True, bbox_to_anchor=(0.5, 0))
#     fig.suptitle(
#         f"Spline fit (multinomial) — Log-odds of cluster membership vs {ref_label}\n"
#         f"Dashed lines = age_group boundaries (30, 45, 60, 75)",
#         fontsize=13, fontweight="bold", color="black",
#     )
#     plt.tight_layout(rect=[0, 0.06, 1, 0.95])
#     plt.savefig(os.path.join(OUT_DIR_SPLINE, f"spline_age_by_cluster_mnlogit_{n_clusters}clusters.png"),
#                 dpi=200, bbox_inches="tight", facecolor="white")
#     plt.close()
#     print(f"Saved: spline_age_by_cluster_mnlogit_{n_clusters}clusters.png")
#
#     # ── FIGURE 2 — Tous les clusters sur un seul graphe ──────────────────────
#     fig, ax = plt.subplots(figsize=(14, 6), facecolor="white")
#     ax.set_facecolor("white")
#     fig.patch.set_facecolor("white")
#
#     for i, (c, color) in enumerate(zip(non_ref_clusters, palette)):
#         c_label = CLUSTER_LABELS.get(c, f"Cluster {c}")
#         ax.plot(age_range, logit_preds[:, i], color=color, linewidth=2, label=c_label)
#
#     for boundary in [30, 45, 60, 75]:
#         ax.axvline(boundary, color="lightgrey", linestyle="--", linewidth=0.8)
#
#     ymax = ax.get_ylim()[1]
#     for boundary in [30, 45, 60, 75]:
#         ax.text(boundary + 0.5, ymax * 0.98, str(boundary),
#                 fontsize=8, color="grey", va="top")
#
#     ax.axhline(0, color="black", linewidth=0.8, linestyle=":")
#     ax.set_xlabel("Age (years)", fontsize=12, color="black")
#     ax.set_ylabel(f"Log-odds vs {ref_label}", fontsize=12, color="black")
#     ax.set_title(
#         f"Spline fit (multinomial) — Log-odds of cluster membership vs {ref_label}\n"
#         f"(dashed lines = age_group boundaries: 30, 45, 60, 75)",
#         fontsize=13, fontweight="bold", color="black", pad=15,
#     )
#     ax.tick_params(colors="black")
#     ax.legend(title=f"Cluster (vs {ref_label})", bbox_to_anchor=(1.02, 1),
#                 loc="upper left", fontsize=8, title_fontsize=9, frameon=True)
#     ax.spines["top"].set_visible(False)
#     ax.spines["right"].set_visible(False)
#
#     plt.tight_layout()
#     plt.savefig(os.path.join(OUT_DIR_SPLINE, f"spline_age_all_clusters_mnlogit_{n_clusters}clusters.png"),
#                 dpi=200, bbox_inches="tight", facecolor="white")
#     plt.close()
#     print(f"Saved: spline_age_all_clusters_mnlogit_{n_clusters}clusters.png")

In [299]:
# # ==============================================================================
# # CHUNK 3bis — Splines binaires — cluster k vs the rest
# # ==============================================================================
#
# for config in CLUSTERING_RUNS:
#     n_clusters     = config['n_clusters']
#     mcs            = config['mcs']
#     ms             = config['ms']
#     CLUSTER_LABELS = config['cluster_labels']
#     CLUSTER_ORDER  = config['cluster_order']
#
#     OUT_DIR_REG    = f"{BASE_DIR}/final_mcs{mcs}_ms{ms}/logistic_regression"
#     OUT_DIR_SPLINE = os.path.join(OUT_DIR_REG, "splines")
#     os.makedirs(OUT_DIR_SPLINE, exist_ok=True)
#
#     CSV_PATH = f"{BASE_DIR}/final_mcs{mcs}_ms{ms}/clustering_{RUN_LABEL}_{SCALER}_mcs{mcs}_ms{ms}_labeled.csv"
#     df_clust = pd.read_csv(CSV_PATH, low_memory=False)
#     df_clust['complaint_category_reg'] = df_clust['complaint_category'].replace(
#         {cat: 'Metabolic_Hematologic_Toxic' for cat in SYSTEMIC_RARE}
#     )
#     cols_needed = ALL_REG_FEATURES_WITH_TRIAGE + ['cluster', 'age']
#     df_reg = df_clust[[c for c in cols_needed if c in df_clust.columns]].copy()
#     df_reg = df_reg.dropna(subset=['cluster', 'age'])
#     df_reg['cluster'] = df_reg['cluster'].astype(int)
#
#     df_spline = df_reg[["age", "cluster"]].dropna()
#     df_spline = df_spline[df_spline["age"].between(14, 120)].copy()
#
#     clusters = [
#         c for label in CLUSTER_ORDER
#         for c, lbl in CLUSTER_LABELS.items()
#         if lbl == label and c in df_spline["cluster"].unique()
#     ]
#     age_range = np.linspace(df_spline["age"].min(), df_spline["age"].max(), 300)
#
#     spline_basis       = dmatrix("cr(age, df=4) - 1", {"age": df_spline["age"]}, return_type="dataframe")
#     spline_basis_range = dmatrix("cr(age, df=4) - 1", {"age": age_range},        return_type="dataframe")
#
#     # ── FIGURE 1 — One subplot per cluster ───────────────────────────────────
#     n_cols = min(3, len(clusters))
#     n_rows = (len(clusters) + n_cols - 1) // n_cols
#
#     fig, axes = plt.subplots(n_rows, n_cols,
#                               figsize=(n_cols * 6, n_rows * 5), facecolor="white")
#     fig.patch.set_facecolor("white")
#     axes = np.array(axes).flatten()
#
#     df_spline["age_bin"] = pd.cut(df_spline["age"], bins=20)
#
#     for i, c in enumerate(clusters):
#         ax      = axes[i]
#         c_label = CLUSTER_LABELS.get(c, f"Cluster {c}")
#         y_bin   = (df_spline["cluster"] == c).astype(int)
#         model   = sm.Logit(y_bin, spline_basis).fit(disp=False)
#
#         logit_pred = spline_basis_range.values @ model.params.values
#         cov        = model.cov_params()
#         gradient   = np.array(spline_basis_range)
#         se         = np.sqrt((gradient @ cov.values @ gradient.T).diagonal())
#         ci_low     = logit_pred - 1.96 * se
#         ci_high    = logit_pred + 1.96 * se
#
#         raw = (
#             df_spline.groupby("age_bin", observed=True)
#             .apply(lambda x: (x["cluster"] == c).mean())
#             .reset_index()
#         )
#         raw["age_mid"]   = raw["age_bin"].apply(lambda x: x.mid)
#         raw["logit_raw"] = np.log(raw[0] / (1 - raw[0]))
#         raw = raw[(raw[0] > 0) & (raw[0] < 1)]
#
#         ax.scatter(raw["age_mid"], raw["logit_raw"],
#                    color="grey", s=20, alpha=0.5, label="Raw proportion (logit)")
#         ax.plot(age_range, logit_pred,
#                 color="steelblue", linewidth=2, label="Spline fit (logit)")
#         ax.fill_between(age_range, ci_low, ci_high,
#                         alpha=0.2, color="steelblue", label="95% CI")
#         for boundary in [30, 45, 60, 75]:
#             ax.axvline(boundary, color="lightgrey", linestyle="--", linewidth=0.8)
#
#         ax.set_facecolor("white")
#         ax.set_title(c_label, fontsize=10, fontweight="bold", color="black")
#         ax.set_xlabel("Age",  fontsize=10, color="black")
#         ax.set_ylabel("Log-odds (logit)", fontsize=10, color="black")
#         ax.tick_params(colors="black")
#         ax.spines["top"].set_visible(False)
#         ax.spines["right"].set_visible(False)
#
#     for j in range(i + 1, len(axes)):
#         axes[j].set_visible(False)
#
#     legend_elements = [
#         Line2D([0], [0], color="grey",      linewidth=0, marker="o",
#                markersize=6, alpha=0.5, label="Raw proportion"),
#         Line2D([0], [0], color="steelblue", linewidth=2, label="Spline fit"),
#         Line2D([0], [0], color="steelblue", linewidth=8, alpha=0.2, label="95% CI"),
#         Line2D([0], [0], color="lightgrey", linewidth=1, linestyle="--",
#                label="Age group boundary"),
#     ]
#     fig.legend(handles=legend_elements, loc="lower center", ncol=4,
#                fontsize=9, frameon=True, bbox_to_anchor=(0.5, 0))
#     fig.suptitle(
#         f"Spline fit — P(cluster membership | age) — {n_clusters} clusters\n"
#         "Dashed lines = age_group boundaries (30, 45, 60, 75)",
#         fontsize=13, fontweight="bold", color="black",
#     )
#     plt.tight_layout(rect=[0, 0.06, 1, 0.95])
#     plt.savefig(os.path.join(OUT_DIR_SPLINE, f"spline_age_by_cluster_{n_clusters}clusters.png"),
#                 dpi=200, bbox_inches="tight", facecolor="white")
#     plt.close()
#     print(f"Saved: spline_age_by_cluster_{n_clusters}clusters.png")
#
#     # ── FIGURE 2 — All clusters on one figure ─────────────────────────────────
#     fig, ax = plt.subplots(figsize=(14, 6), facecolor="white")
#     ax.set_facecolor("white")
#     fig.patch.set_facecolor("white")
#     palette = sns.color_palette("tab10", len(clusters))
#
#     for c, color in zip(clusters, palette):
#         c_label    = CLUSTER_LABELS.get(c, f"Cluster {c}")
#         y_bin      = (df_spline["cluster"] == c).astype(int)
#         model      = sm.Logit(y_bin, spline_basis).fit(disp=False)
#         logit_pred = spline_basis_range.values @ model.params.values
#         ax.plot(age_range, logit_pred, color=color, linewidth=2, label=c_label)
#
#     for boundary in [30, 45, 60, 75]:
#         ax.axvline(boundary, color="lightgrey", linestyle="--", linewidth=0.8)
#
#     ymax = ax.get_ylim()[1]
#     for boundary in [30, 45, 60, 75]:
#         ax.text(boundary + 0.5, ymax * 0.98, str(boundary),
#                 fontsize=8, color="grey", va="top")
#
#     ax.set_xlabel("Age (years)",      fontsize=12, color="black")
#     ax.set_ylabel("Log-odds (logit)", fontsize=12, color="black")
#     ax.set_title(
#         f"Spline fit — Log-odds of cluster membership by age — {n_clusters} clusters\n"
#         "(dashed lines = age_group boundaries: 30, 45, 60, 75)",
#         fontsize=13, fontweight="bold", color="black", pad=15,
#     )
#     ax.tick_params(colors="black")
#     ax.legend(title="Cluster", bbox_to_anchor=(1.02, 1),
#               loc="upper left", fontsize=8, title_fontsize=9, frameon=True)
#     ax.spines["top"].set_visible(False)
#     ax.spines["right"].set_visible(False)
#
#     plt.tight_layout()
#     plt.savefig(os.path.join(OUT_DIR_SPLINE, f"spline_age_all_clusters_{n_clusters}clusters.png"),
#                 dpi=200, bbox_inches="tight", facecolor="white")
#     plt.close()
#     print(f"Saved: spline_age_all_clusters_{n_clusters}clusters.png")

splines with traige grouped

In [300]:
# ==============================================================================
# CHUNK 3 — Splines — age vs cluster membership (multinomial vs ref cluster)
# ==============================================================================

from matplotlib.lines import Line2D

for config in CLUSTERING_RUNS:
    n_clusters     = config['n_clusters']
    mcs            = config['mcs']
    ms             = config['ms']
    ref_c          = config['ref_cluster']
    CLUSTER_LABELS = config['cluster_labels']
    CLUSTER_ORDER  = config['cluster_order']

    # ── Paths ─────────────────────────────────────────────────────────────────
    CSV_PATH    = f"{BASE_DIR}/final_mcs{mcs}_ms{ms}/clustering_{RUN_LABEL}_{SCALER}_mcs{mcs}_ms{ms}_labeled.csv"
    OUT_DIR_REG = f"{BASE_DIR}/final_mcs{mcs}_ms{ms}/logistic_regression"
    OUT_DIR_SPLINE = os.path.join(OUT_DIR_REG, "splines")
    os.makedirs(OUT_DIR_SPLINE, exist_ok=True)

    # ── Load + remaps ─────────────────────────────────────────────────────────
    df_clust = pd.read_csv(CSV_PATH, low_memory=False)

    for col, mapping in VITAL_SIGN_REMAP.items():
        if col in df_clust.columns:
            df_clust[col] = df_clust[col].map(mapping)

    df_clust['triage_grouped'] = (
        df_clust['triage'].astype(float).astype(int)
        .map(TRIAGE_GROUP_MAP).astype(str)
    )
    df_clust['complaint_category_reg'] = df_clust['complaint_category'].replace(
        {cat: 'Other' for cat in SYSTEMIC_RARE}
    )
    df_clust['cluster_label'] = df_clust['cluster'].map(CLUSTER_LABELS)
    df_clust['cluster_label'] = pd.Categorical(
        df_clust['cluster_label'], categories=CLUSTER_ORDER, ordered=True
    )

    # ── Build df_reg ──────────────────────────────────────────────────────────
    cols_needed = list(set(
        ALL_REG_FEATURES_WITH_TRIAGE + ALL_REG_FEATURES_NO_TRIAGE +
        ['cluster', 'cluster_label', 'age']
    ))
    df_reg = df_clust[[c for c in cols_needed if c in df_clust.columns]].copy()
    df_reg = df_reg.dropna(subset=ALL_REG_FEATURES_WITH_TRIAGE + ['cluster'])
    df_reg['cluster'] = df_reg['cluster'].astype(int)
    df_reg['triage_grouped'] = df_reg['triage_grouped'].astype(str)
    df_reg['age_group'] = df_reg['age_group'].astype(str)
    df_reg['sex'] = df_reg['sex'].astype(str)
    df_reg['transport_grouped'] = df_reg['transport_grouped'].astype(str)
    df_reg['complaint_category_reg'] = df_reg['complaint_category_reg'].astype(str)
    for col in REG_FEATURES_STATUS_CLEAN:
        if col in df_reg.columns:
            df_reg[col] = df_reg[col].astype(str)
    df_reg['is_outlier'] = (df_reg['cluster'] == -1).astype(int)

    print(f"\n{'='*60}")
    print(f"RUN — {n_clusters} clusters (mcs={mcs}, ms={ms})")
    print(f"{'='*60}")
    print(f"Regression dataset: {len(df_reg):,} patients")

    # ── Spline setup ──────────────────────────────────────────────────────────
    ref_c_spline = ref_c
    ref_label    = CLUSTER_LABELS[ref_c_spline]

    df_spline = df_reg[["age", "cluster"]].dropna()
    df_spline = df_spline[df_spline["age"].between(14, 120)].copy()

    df_spline["cluster"] = pd.Categorical(
        df_spline["cluster"],
        categories=[ref_c_spline] + sorted([c for c in df_spline["cluster"].unique() if c != ref_c_spline])
    )

    age_range = np.linspace(df_spline["age"].min(), df_spline["age"].max(), 300)

    spline_basis       = dmatrix("cr(age, df=4) - 1", {"age": df_spline["age"]}, return_type="dataframe")
    spline_basis_range = dmatrix("cr(age, df=4) - 1", {"age": age_range},        return_type="dataframe")
    spline_basis_model = sm.add_constant(spline_basis)
    spline_basis_pred  = sm.add_constant(spline_basis_range)

    # ── MNLogit ───────────────────────────────────────────────────────────────
    y_cat        = df_spline["cluster"].cat.codes
    model        = sm.MNLogit(y_cat, spline_basis_model).fit(method="newton", maxiter=500, disp=False)

    non_ref_clusters = [
        c for label in CLUSTER_ORDER
        for c, lbl in CLUSTER_LABELS.items()
        if lbl == label and c != ref_c_spline and c in df_spline["cluster"].unique()
    ]
    n_outcomes   = len(non_ref_clusters)
    logit_preds  = spline_basis_pred.values @ model.params.values

    # ── CI ────────────────────────────────────────────────────────────────────
    cov      = model.cov_params()
    n_params = spline_basis_pred.shape[1]
    ci_lows, ci_highs = [], []

    for k in range(n_outcomes):
        idx   = slice(k * n_params, (k + 1) * n_params)
        cov_k = cov.values[idx, idx]
        grad  = spline_basis_pred.values
        se_k  = np.sqrt((grad @ cov_k @ grad.T).diagonal())
        ci_lows.append(logit_preds[:, k] - 1.96 * se_k)
        ci_highs.append(logit_preds[:, k] + 1.96 * se_k)

    # ── FIGURE 1 — One subplot per cluster ───────────────────────────────────
    palette = sns.color_palette("tab10", n_outcomes)
    n_cols  = min(3, n_outcomes)
    n_rows  = (n_outcomes + n_cols - 1) // n_cols

    fig, axes = plt.subplots(n_rows, n_cols,
                              figsize=(n_cols * 6, n_rows * 5), facecolor="white")
    fig.patch.set_facecolor("white")
    axes = np.array(axes).flatten()

    df_spline["age_bin"] = pd.cut(df_spline["age"], bins=20)

    for i, (c, color) in enumerate(zip(non_ref_clusters, palette)):
        ax      = axes[i]
        c_label = CLUSTER_LABELS.get(c, f"Cluster {c}")

        df_pair = df_spline[df_spline["cluster"].isin([c, ref_c_spline])]
        raw = (
            df_pair.groupby("age_bin", observed=True)
            .apply(lambda x: (x["cluster"] == c).mean())
            .reset_index()
        )
        raw["age_mid"]   = raw["age_bin"].apply(lambda x: x.mid)
        raw["logit_raw"] = np.log(raw[0] / (1 - raw[0]))
        raw = raw[(raw[0] > 0) & (raw[0] < 1)]

        ax.scatter(raw["age_mid"], raw["logit_raw"],
                   color="grey", s=20, alpha=0.5, label="Raw proportion (logit)")
        ax.plot(age_range, logit_preds[:, i],
                color=color, linewidth=2, label="Spline fit")
        ax.fill_between(age_range, ci_lows[i], ci_highs[i],
                        alpha=0.2, color=color, label="95% CI")

        for boundary in [30, 45, 60, 75]:
            ax.axvline(boundary, color="lightgrey", linestyle="--", linewidth=0.8)

        ax.set_facecolor("white")
        ax.set_title(f"{c_label}\nvs {ref_label} (ref)",
                     fontsize=10, fontweight="bold", color="black")
        ax.set_xlabel("Age", fontsize=10, color="black")
        ax.set_ylabel(f"Log-odds vs {ref_label}", fontsize=9, color="black")
        ax.tick_params(colors="black")
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    legend_elements = [
        Line2D([0], [0], color="grey",      linewidth=0, marker="o",
               markersize=6, alpha=0.5, label="Raw proportion (logit)"),
        Line2D([0], [0], color="black",     linewidth=2,  label="Spline fit"),
        Line2D([0], [0], color="black",     linewidth=8,  alpha=0.2, label="95% CI"),
        Line2D([0], [0], color="lightgrey", linewidth=1,  linestyle="--", label="Age group boundary"),
    ]
    fig.legend(handles=legend_elements, loc="lower center", ncol=4,
               fontsize=9, frameon=True, bbox_to_anchor=(0.5, 0))
    fig.suptitle(
        f"Spline fit (multinomial) — Log-odds of cluster membership vs {ref_label}\n"
        f"Dashed lines = age_group boundaries (30, 45, 60, 75)",
        fontsize=13, fontweight="bold", color="black",
    )
    plt.tight_layout(rect=[0, 0.06, 1, 0.95])
    plt.savefig(os.path.join(OUT_DIR_SPLINE, f"spline_age_by_cluster_mnlogit_{n_clusters}clusters.png"),
                dpi=200, bbox_inches="tight", facecolor="white")
    plt.close()
    print(f"Saved: spline_age_by_cluster_mnlogit_{n_clusters}clusters.png")

    # ── FIGURE 2 — All clusters on one figure ─────────────────────────────────
    fig, ax = plt.subplots(figsize=(14, 6), facecolor="white")
    ax.set_facecolor("white")
    fig.patch.set_facecolor("white")

    for i, (c, color) in enumerate(zip(non_ref_clusters, palette)):
        c_label = CLUSTER_LABELS.get(c, f"Cluster {c}")
        ax.plot(age_range, logit_preds[:, i], color=color, linewidth=2, label=c_label)

    for boundary in [30, 45, 60, 75]:
        ax.axvline(boundary, color="lightgrey", linestyle="--", linewidth=0.8)

    ymax = ax.get_ylim()[1]
    for boundary in [30, 45, 60, 75]:
        ax.text(boundary + 0.5, ymax * 0.98, str(boundary),
                fontsize=8, color="grey", va="top")

    ax.axhline(0, color="black", linewidth=0.8, linestyle=":")
    ax.set_xlabel("Age (years)", fontsize=12, color="black")
    ax.set_ylabel(f"Log-odds vs {ref_label}", fontsize=12, color="black")
    ax.set_title(
        f"Spline fit (multinomial) — Log-odds of cluster membership vs {ref_label}\n"
        f"(dashed lines = age_group boundaries: 30, 45, 60, 75)",
        fontsize=13, fontweight="bold", color="black", pad=15,
    )
    ax.tick_params(colors="black")
    ax.legend(title=f"Cluster (vs {ref_label})", bbox_to_anchor=(1.02, 1),
              loc="upper left", fontsize=8, title_fontsize=9, frameon=True)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR_SPLINE, f"spline_age_all_clusters_mnlogit_{n_clusters}clusters.png"),
                dpi=200, bbox_inches="tight", facecolor="white")
    plt.close()
    print(f"Saved: spline_age_all_clusters_mnlogit_{n_clusters}clusters.png")

KeyboardInterrupt: 

In [ ]:
# ==============================================================================
# CHUNK 3bis — Binary splines — cluster k vs the rest
# ==============================================================================

for config in CLUSTERING_RUNS:
    n_clusters     = config['n_clusters']
    mcs            = config['mcs']
    ms             = config['ms']
    CLUSTER_LABELS = config['cluster_labels']
    CLUSTER_ORDER  = config['cluster_order']

    OUT_DIR_REG    = f"{BASE_DIR}/final_mcs{mcs}_ms{ms}/logistic_regression"
    OUT_DIR_SPLINE = os.path.join(OUT_DIR_REG, "splines")
    os.makedirs(OUT_DIR_SPLINE, exist_ok=True)

    # ── Load + remaps ─────────────────────────────────────────────────────────
    CSV_PATH = f"{BASE_DIR}/final_mcs{mcs}_ms{ms}/clustering_{RUN_LABEL}_{SCALER}_mcs{mcs}_ms{ms}_labeled.csv"
    df_clust = pd.read_csv(CSV_PATH, low_memory=False)

    for col, mapping in VITAL_SIGN_REMAP.items():
        if col in df_clust.columns:
            df_clust[col] = df_clust[col].map(mapping)

    df_clust['triage_grouped'] = (
        df_clust['triage'].astype(float).astype(int)
        .map(TRIAGE_GROUP_MAP).astype(str)
    )
    df_clust['complaint_category_reg'] = df_clust['complaint_category'].replace(
        {cat: 'Other' for cat in SYSTEMIC_RARE}
    )

    # ── Build df_reg ──────────────────────────────────────────────────────────
    cols_needed = list(set(
        ALL_REG_FEATURES_WITH_TRIAGE + ALL_REG_FEATURES_NO_TRIAGE +
        ['cluster', 'age']
    ))
    df_reg = df_clust[[c for c in cols_needed if c in df_clust.columns]].copy()
    df_reg = df_reg.dropna(subset=['cluster', 'age'])
    df_reg['cluster'] = df_reg['cluster'].astype(int)

    # ── Spline setup ──────────────────────────────────────────────────────────
    df_spline = df_reg[["age", "cluster"]].dropna()
    df_spline = df_spline[df_spline["age"].between(14, 120)].copy()

    clusters = [
        c for label in CLUSTER_ORDER
        for c, lbl in CLUSTER_LABELS.items()
        if lbl == label and c in df_spline["cluster"].unique()
    ]
    age_range = np.linspace(df_spline["age"].min(), df_spline["age"].max(), 300)

    spline_basis       = dmatrix("cr(age, df=4) - 1", {"age": df_spline["age"]}, return_type="dataframe")
    spline_basis_range = dmatrix("cr(age, df=4) - 1", {"age": age_range},        return_type="dataframe")

    # ── FIGURE 1 — One subplot per cluster ───────────────────────────────────
    n_cols = min(3, len(clusters))
    n_rows = (len(clusters) + n_cols - 1) // n_cols

    fig, axes = plt.subplots(n_rows, n_cols,
                              figsize=(n_cols * 6, n_rows * 5), facecolor="white")
    fig.patch.set_facecolor("white")
    axes = np.array(axes).flatten()

    df_spline["age_bin"] = pd.cut(df_spline["age"], bins=20)

    for i, c in enumerate(clusters):
        ax      = axes[i]
        c_label = CLUSTER_LABELS.get(c, f"Cluster {c}")
        y_bin   = (df_spline["cluster"] == c).astype(int)
        model   = sm.Logit(y_bin, spline_basis).fit(disp=False)

        logit_pred = spline_basis_range.values @ model.params.values
        cov        = model.cov_params()
        gradient   = np.array(spline_basis_range)
        se         = np.sqrt((gradient @ cov.values @ gradient.T).diagonal())
        ci_low     = logit_pred - 1.96 * se
        ci_high    = logit_pred + 1.96 * se

        raw = (
            df_spline.groupby("age_bin", observed=True)
            .apply(lambda x: (x["cluster"] == c).mean())
            .reset_index()
        )
        raw["age_mid"]   = raw["age_bin"].apply(lambda x: x.mid)
        raw["logit_raw"] = np.log(raw[0] / (1 - raw[0]))
        raw = raw[(raw[0] > 0) & (raw[0] < 1)]

        ax.scatter(raw["age_mid"], raw["logit_raw"],
                   color="grey", s=20, alpha=0.5, label="Raw proportion (logit)")
        ax.plot(age_range, logit_pred,
                color="steelblue", linewidth=2, label="Spline fit (logit)")
        ax.fill_between(age_range, ci_low, ci_high,
                        alpha=0.2, color="steelblue", label="95% CI")
        for boundary in [30, 45, 60, 75]:
            ax.axvline(boundary, color="lightgrey", linestyle="--", linewidth=0.8)

        ax.set_facecolor("white")
        ax.set_title(c_label, fontsize=10, fontweight="bold", color="black")
        ax.set_xlabel("Age",  fontsize=10, color="black")
        ax.set_ylabel("Log-odds (logit)", fontsize=10, color="black")
        ax.tick_params(colors="black")
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    legend_elements = [
        Line2D([0], [0], color="grey",      linewidth=0, marker="o",
               markersize=6, alpha=0.5, label="Raw proportion"),
        Line2D([0], [0], color="steelblue", linewidth=2, label="Spline fit"),
        Line2D([0], [0], color="steelblue", linewidth=8, alpha=0.2, label="95% CI"),
        Line2D([0], [0], color="lightgrey", linewidth=1, linestyle="--",
               label="Age group boundary"),
    ]
    fig.legend(handles=legend_elements, loc="lower center", ncol=4,
               fontsize=9, frameon=True, bbox_to_anchor=(0.5, 0))
    fig.suptitle(
        f"Spline fit — P(cluster membership | age) — {n_clusters} clusters\n"
        "Dashed lines = age_group boundaries (30, 45, 60, 75)",
        fontsize=13, fontweight="bold", color="black",
    )
    plt.tight_layout(rect=[0, 0.06, 1, 0.95])
    plt.savefig(os.path.join(OUT_DIR_SPLINE, f"spline_age_by_cluster_{n_clusters}clusters.png"),
                dpi=200, bbox_inches="tight", facecolor="white")
    plt.close()
    print(f"Saved: spline_age_by_cluster_{n_clusters}clusters.png")

    # ── FIGURE 2 — All clusters on one figure ─────────────────────────────────
    fig, ax = plt.subplots(figsize=(14, 6), facecolor="white")
    ax.set_facecolor("white")
    fig.patch.set_facecolor("white")
    palette = sns.color_palette("tab10", len(clusters))

    for c, color in zip(clusters, palette):
        c_label    = CLUSTER_LABELS.get(c, f"Cluster {c}")
        y_bin      = (df_spline["cluster"] == c).astype(int)
        model      = sm.Logit(y_bin, spline_basis).fit(disp=False)
        logit_pred = spline_basis_range.values @ model.params.values
        ax.plot(age_range, logit_pred, color=color, linewidth=2, label=c_label)

    for boundary in [30, 45, 60, 75]:
        ax.axvline(boundary, color="lightgrey", linestyle="--", linewidth=0.8)

    ymax = ax.get_ylim()[1]
    for boundary in [30, 45, 60, 75]:
        ax.text(boundary + 0.5, ymax * 0.98, str(boundary),
                fontsize=8, color="grey", va="top")

    ax.set_xlabel("Age (years)",      fontsize=12, color="black")
    ax.set_ylabel("Log-odds (logit)", fontsize=12, color="black")
    ax.set_title(
        f"Spline fit — Log-odds of cluster membership by age — {n_clusters} clusters\n"
        "(dashed lines = age_group boundaries: 30, 45, 60, 75)",
        fontsize=13, fontweight="bold", color="black", pad=15,
    )
    ax.tick_params(colors="black")
    ax.legend(title="Cluster", bbox_to_anchor=(1.02, 1),
              loc="upper left", fontsize=8, title_fontsize=9, frameon=True)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR_SPLINE, f"spline_age_all_clusters_{n_clusters}clusters.png"),
                dpi=200, bbox_inches="tight", facecolor="white")
    plt.close()
    print(f"Saved: spline_age_all_clusters_{n_clusters}clusters.png")


Cluster 4 (brun) — forte décroissance avec l'âge
C'est ton cluster "passage simple / aucun examen". Les jeunes (15-30 ans) y sont massivement représentés (~30%) et cette probabilité chute fortement après 45 ans. Les jeunes adultes consultent aux urgences pour des motifs simples ne nécessitant pas d'explorations.
Cluster 7 (jaune-vert) — décroissance progressive
Surreprésenté chez les jeunes (~20%) et quasi absent après 75 ans. Probablement lié à des motifs traumatologiques ou musculo-squelettiques, typiques des jeunes actifs.
Cluster 5 (rose) — forte croissance avec l'âge
Probabilité qui monte fortement après 60 ans et explose après 75 ans (~30%). C'est ton cluster de prise en charge lourde des patients âgés.
Cluster 3 (violet) — légère croissance
Stable avec une légère augmentation après 60 ans — probablement les bilans cardiovasculaires/neurologiques plus fréquents avec l'âge.
Clusters 0 et 1 (orange, vert) — croissance modérée
Augmentent progressivement avec l'âge, pic vers 70-80 ans puis redescendent légèrement. Bilans biologiques et imagerie plus fréquents chez les patients d'âge moyen à âgés.
Cluster 6 (gris) — en cloche
Pic vers 70-80 ans puis décroissance — profil typique des patients âgés mais pas très vieux.
Cluster 2 (rouge) et Outliers (bleu) — en cloche tardive
Pic vers 70-80 ans, profils atypiques ou complexes plus fréquents chez les personnes âgées.
Cluster 8 (cyan) — décroissance
Surreprésenté chez les jeunes, décroît avec l'âge.

Ce que ça justifie pour ta régression :
Les courbes sont clairement non-linéaires — elles montent, descendent, ont des inflexions — ce qui justifie pleinement age_group en catégoriel plutôt qu'age en continu. Et les inflexions coïncident globalement avec tes frontières à 30, 45, 60 et 75 ans, ce qui valide tes coupures a priori.

In [ ]:
# ==============================================================================
# CHUNK 4 — Univariate multinomial logistic regressions
# ==============================================================================

for config in CLUSTERING_RUNS:
    n_clusters     = config['n_clusters']
    mcs            = config['mcs']
    ms             = config['ms']
    ref_c          = config['ref_cluster']
    CLUSTER_LABELS = config['cluster_labels']
    CLUSTER_ORDER  = config['cluster_order']
    OUT_DIR_REG    = f"{BASE_DIR}/final_mcs{mcs}_ms{ms}/logistic_regression"
    os.makedirs(OUT_DIR_REG, exist_ok=True)

    # ── Load + remaps ─────────────────────────────────────────────────────────
    CSV_PATH = f"{BASE_DIR}/final_mcs{mcs}_ms{ms}/clustering_{RUN_LABEL}_{SCALER}_mcs{mcs}_ms{ms}_labeled.csv"
    df_clust = pd.read_csv(CSV_PATH, low_memory=False)

    for col, mapping in VITAL_SIGN_REMAP.items():
        if col in df_clust.columns:
            df_clust[col] = df_clust[col].map(mapping)

    df_clust['triage_grouped'] = (
        df_clust['triage'].astype(float).astype(int)
        .map(TRIAGE_GROUP_MAP).astype(str)
    )
    df_clust['complaint_category_reg'] = df_clust['complaint_category'].replace(
        {cat: 'Other' for cat in SYSTEMIC_RARE}
    )
    df_clust['cluster_label'] = df_clust['cluster'].map(CLUSTER_LABELS)
    df_clust['cluster_label'] = pd.Categorical(
        df_clust['cluster_label'], categories=CLUSTER_ORDER, ordered=True
    )

    # ── Build df_reg ──────────────────────────────────────────────────────────
    cols_needed = list(set(
        ALL_REG_FEATURES_WITH_TRIAGE + ALL_REG_FEATURES_NO_TRIAGE +
        ['cluster', 'cluster_label', 'age']
    ))
    df_reg = df_clust[[c for c in cols_needed if c in df_clust.columns]].copy()
    df_reg = df_reg.dropna(subset=ALL_REG_FEATURES_WITH_TRIAGE + ['cluster'])
    df_reg['cluster']                = df_reg['cluster'].astype(int)
    df_reg['triage_grouped']         = df_reg['triage_grouped'].astype(str)
    df_reg['age_group']              = df_reg['age_group'].astype(str)
    df_reg['sex']                    = df_reg['sex'].astype(str)
    df_reg['transport_grouped']      = df_reg['transport_grouped'].astype(str)
    df_reg['complaint_category_reg'] = df_reg['complaint_category_reg'].astype(str)
    for col in REG_FEATURES_STATUS_CLEAN:
        if col in df_reg.columns:
            df_reg[col] = df_reg[col].astype(str)
    df_reg['is_outlier'] = (df_reg['cluster'] == -1).astype(int)

    print("\n" + "="*60)
    print(f"UNIVARIATE REGRESSIONS — {n_clusters} clusters")
    print("="*60)
    print(f"Regression dataset: {len(df_reg):,} patients")

    all_uni_res = []

    for var in ALL_REG_FEATURES_WITH_TRIAGE:
        if var not in df_reg.columns:
            print(f"MISSING  {var}")
            continue

        ref_val = REF_CATEGORIES.get(var, df_reg[var].mode()[0])
        formula = f"cluster ~ C({var}, Treatment('{ref_val}'))"
        print(f"Trying: {formula}")

        model = None
        for method in ["bfgs", "lbfgs", "cg", "newton"]:
            try:
                model = mnlogit(formula, data=df_reg).fit(
                    method  = method,
                    maxiter = 2000,
                    gtol    = 1e-5,
                    disp    = False,
                )
                if not model.mle_retvals.get("converged", True):
                    print(f"WARNING {var} (method={method}): converged=False")
                else:
                    print(f"OK  {var} (method={method})")
                break
            except Exception as e:
                print(f"  {method} failed: {e}")

        if model is None:
            print(f"FAILED  {var} — all methods failed")
            continue

        df_res = format_mnlogit_results(model, ref_c, CLUSTER_LABELS)
        df_res["model"]   = "univariate"
        df_res["feature"] = var
        all_uni_res.append(df_res)

    # ── Concat + CSV + forest plots ───────────────────────────────────────────
    if len(all_uni_res) == 0:
        print("WARNING: no model converged")
    else:
        df_univariate = pd.concat(all_uni_res, ignore_index=True)
        df_univariate = df_univariate.sort_values(
            ["cluster_vs_ref", "feature"]
        ).reset_index(drop=True)

        # ── Flag unstable estimates ───────────────────────────────────────────
        df_univariate["unstable"] = (
            (df_univariate["CI_low"] > df_univariate["OR"]) |
            (df_univariate["CI_high"] < df_univariate["OR"])
        )

        df_univariate.to_csv(
            os.path.join(OUT_DIR_REG, f"univariate_results_{n_clusters}clusters.csv"),
            index=False
        )
        print(f"\nSaved: univariate_results_{n_clusters}clusters.csv ({len(df_univariate)} rows)")

        n_unstable = df_univariate["unstable"].sum()
        if n_unstable > 0:
            print(f"WARNING — {n_unstable} unstable estimates (quasi-separation) flagged in CSV")
            print(df_univariate[df_univariate["unstable"]][
                ["cluster_vs_ref", "variable", "OR", "CI_low", "CI_high", "p_value"]
            ])

        ref_label       = CLUSTER_LABELS.get(ref_c, f"C{ref_c}")
        clusters_vs_ref = sorted(df_univariate["cluster_vs_ref"].unique())

        # ── Helper local — draw forest plot ───────────────────────────────────
        def draw_forest(ax, df_plot, colors, point_size=40):
            df_plot = df_plot.copy().reset_index(drop=True)
            n = len(df_plot)

            # ── Fix unstable CI for display only ──────────────────────────────
            ci_low_disp  = df_plot.apply(
                lambda r: r["OR"] * 0.5 if r["unstable"] else r["CI_low"], axis=1
            ).values
            ci_high_disp = df_plot.apply(
                lambda r: r["OR"] * 2.0 if r["unstable"] else r["CI_high"], axis=1
            ).values
            or_vals = df_plot["OR"].values

            for i in range(n):
                if i % 2 == 0:
                    ax.axhspan(i - 0.5, i + 0.5, color="lightgrey", alpha=0.3, zorder=0)

            for i in range(n):
                ax.plot(
                    [ci_low_disp[i], ci_high_disp[i]],
                    [i, i],
                    color=colors[i],
                    linewidth=1.5,
                    zorder=1,
                    linestyle="--" if df_plot["unstable"].iloc[i] else "-"
                )
                # Triangle pour les estimations instables, point sinon
                marker = "^" if df_plot["unstable"].iloc[i] else "o"
                ax.scatter(
                    or_vals[i],
                    i,
                    color=colors[i],
                    s=point_size,
                    zorder=2,
                    marker=marker
                )

            ax.axvline(1, color="black", linewidth=0.8, linestyle="-")
            for x in [0.5, 2.0]:
                ax.axvline(x, color="grey", linewidth=0.6, linestyle="--", alpha=0.6)
            ax.set_xscale("log")
            ax.set_yticks(range(n))
            ax.spines["top"].set_visible(False)
            ax.spines["right"].set_visible(False)

        # ── Forest plot — one plot per cluster_vs_ref ─────────────────────────
        for clust_label in clusters_vs_ref:
            df_plot = (
                df_univariate[df_univariate["cluster_vs_ref"] == clust_label]
                .sort_values("OR", ascending=True)
                .reset_index(drop=True)
            )
            n_vars = len(df_plot)
            colors = [
                "crimson"   if (s and or_val > 1)  else
                "steelblue" if (s and or_val <= 1) else
                "lightgrey"
                for s, or_val in zip(df_plot["significant"], df_plot["OR"])
            ]

            fig, ax = plt.subplots(figsize=(8, max(6, n_vars * 0.35)), facecolor="white")
            ax.set_facecolor("white")

            draw_forest(ax, df_plot, colors, point_size=40)

            ax.set_yticklabels(df_plot["variable"], fontsize=9)
            ax.set_xlabel("Odds Ratio (log scale)", fontsize=11)

            n_unstable_clust = df_plot["unstable"].sum()
            unstable_note = f"\n(▲ = unstable estimate, quasi-separation)" if n_unstable_clust > 0 else ""
            ax.set_title(
                f"{clust_label}\nref = {ref_label}{unstable_note}",
                fontsize=11, fontweight="bold", color="black"
            )
            plt.tight_layout()

            fname_clean = (
                clust_label
                .replace(" ", "_").replace("—", "-")
                .replace("+", "plus").replace("/", "-")
                .replace("(", "").replace(")", "")
            )
            fname = f"forest_plot_univariate_{fname_clean}_{n_clusters}clusters.png"
            plt.savefig(os.path.join(OUT_DIR_REG, fname),
                        dpi=200, bbox_inches="tight", facecolor="white")
            plt.close()
            print(f"Saved: {fname}")

        # ── Combined forest plot — all clusters ───────────────────────────────
        n_clusters_plot = len(clusters_vs_ref)
        n_cols_fp = min(3, n_clusters_plot)
        n_rows_fp = (n_clusters_plot + n_cols_fp - 1) // n_cols_fp

        fig, axes = plt.subplots(
            n_rows_fp, n_cols_fp,
            figsize=(n_cols_fp * 8, n_rows_fp * max(6, n_vars * 0.35))
        )
        axes = np.array(axes).flatten()

        for ax_i, clust_label in enumerate(clusters_vs_ref):
            ax = axes[ax_i]
            df_plot = (
                df_univariate[df_univariate["cluster_vs_ref"] == clust_label]
                .sort_values("OR", ascending=True)
                .reset_index(drop=True)
            )
            n_vars_i = len(df_plot)
            colors = [
                "crimson"   if (s and or_val > 1)  else
                "steelblue" if (s and or_val <= 1) else
                "lightgrey"
                for s, or_val in zip(df_plot["significant"], df_plot["OR"])
            ]

            draw_forest(ax, df_plot, colors, point_size=30)

            ax.set_yticklabels(df_plot["variable"], fontsize=7)
            ax.set_xlabel("OR (log scale)", fontsize=9)
            ax.set_title(clust_label, fontsize=8, fontweight="bold")

        for j in range(ax_i + 1, len(axes)):
            axes[j].set_visible(False)

        plt.suptitle(
            f"Univariate OR — all clusters vs {ref_label} — {n_clusters} clusters\n"
            f"(crimson = sig OR>1 | steelblue = sig OR<1 | grey = ns | ▲ = unstable)",
            fontsize=13, fontweight="bold", y=1.01
        )
        plt.tight_layout()
        plt.savefig(
            os.path.join(OUT_DIR_REG, f"forest_plot_univariate_all_clusters_{n_clusters}clusters.png"),
            dpi=200, bbox_inches="tight"
        )
        plt.close()
        print(f"Saved: forest_plot_univariate_all_clusters_{n_clusters}clusters.png")

In [ ]:
# ==============================================================================
# CHUNK 4b — Univariate binary logistic regressions — Outliers (-1) vs rest
# ==============================================================================

import statsmodels.formula.api as smf

for config in CLUSTERING_RUNS:
    n_clusters     = config['n_clusters']
    mcs            = config['mcs']
    ms             = config['ms']
    ref_c          = config['ref_cluster']
    CLUSTER_LABELS = config['cluster_labels']
    CLUSTER_ORDER  = config['cluster_order']
    OUT_DIR_REG    = f"{BASE_DIR}/final_mcs{mcs}_ms{ms}/logistic_regression"
    os.makedirs(OUT_DIR_REG, exist_ok=True)

    # ── Load + remaps ─────────────────────────────────────────────────────────
    CSV_PATH = f"{BASE_DIR}/final_mcs{mcs}_ms{ms}/clustering_{RUN_LABEL}_{SCALER}_mcs{mcs}_ms{ms}_labeled.csv"
    df_clust = pd.read_csv(CSV_PATH, low_memory=False)

    for col, mapping in VITAL_SIGN_REMAP.items():
        if col in df_clust.columns:
            df_clust[col] = df_clust[col].map(mapping)

    df_clust['triage_grouped'] = (
        df_clust['triage'].astype(float).astype(int)
        .map(TRIAGE_GROUP_MAP).astype(str)
    )
    df_clust['complaint_category_reg'] = df_clust['complaint_category'].replace(
        {cat: 'Other' for cat in SYSTEMIC_RARE}
    )

    # ── Build df_reg ──────────────────────────────────────────────────────────
    cols_needed = list(set(
        ALL_REG_FEATURES_WITH_TRIAGE + ALL_REG_FEATURES_NO_TRIAGE +
        ['cluster', 'age']
    ))
    df_reg = df_clust[[c for c in cols_needed if c in df_clust.columns]].copy()
    df_reg = df_reg.dropna(subset=ALL_REG_FEATURES_WITH_TRIAGE + ['cluster'])
    df_reg['cluster']                = df_reg['cluster'].astype(int)
    df_reg['triage_grouped']         = df_reg['triage_grouped'].astype(str)
    df_reg['age_group']              = df_reg['age_group'].astype(str)
    df_reg['sex']                    = df_reg['sex'].astype(str)
    df_reg['transport_grouped']      = df_reg['transport_grouped'].astype(str)
    df_reg['complaint_category_reg'] = df_reg['complaint_category_reg'].astype(str)
    for col in REG_FEATURES_STATUS_CLEAN:
        if col in df_reg.columns:
            df_reg[col] = df_reg[col].astype(str)
    df_reg['is_outlier'] = (df_reg['cluster'] == -1).astype(int)

    print("\n" + "="*60)
    print(f"UNIVARIATE REGRESSIONS — OUTLIERS vs REST — {n_clusters} clusters")
    print("="*60)
    print(f"Outliers: {df_reg['is_outlier'].sum()} / {len(df_reg)}")

    all_uni_res_outlier = []

    for var in ALL_REG_FEATURES_WITH_TRIAGE:
        if var not in df_reg.columns:
            print(f"MISSING  {var}")
            continue

        ref_val = REF_CATEGORIES.get(var, df_reg[var].mode()[0])
        formula = f"is_outlier ~ C({var}, Treatment('{ref_val}'))"
        print(f"Trying: {formula}")

        model = None
        for method in ["bfgs", "lbfgs", "cg", "newton"]:
            try:
                model = smf.logit(formula, data=df_reg).fit(
                    method  = method,
                    maxiter = 2000,
                    gtol    = 1e-5,
                    disp    = False,
                )
                if not model.mle_retvals.get("converged", True):
                    print(f"WARNING {var} (method={method}): converged=False")
                else:
                    print(f"OK  {var} (method={method})")
                break
            except Exception as e:
                print(f"  {method} failed: {e}")

        if model is None:
            print(f"FAILED  {var} — all methods failed")
            continue

        params = model.params
        conf   = model.conf_int()
        pvals  = model.pvalues
        conf.columns = ["CI_low", "CI_high"]

        rows = []
        for v in params.index:
            if v == "Intercept":
                continue
            clean_var = re.sub(
                r"C\((.+?),\s*Treatment\(.+?\)\)\[T\.(.+?)\]",
                r"\1 = \2", v
            )
            rows.append({
                "variable"   : clean_var,
                "OR"         : round(np.exp(params[v]), 3),
                "CI_low"     : round(np.exp(conf.loc[v, "CI_low"]),  3),
                "CI_high"    : round(np.exp(conf.loc[v, "CI_high"]), 3),
                "p_value"    : round(pvals[v], 4),
                "significant": pvals[v] < 0.05,
                "feature"    : var,
                "model"      : "univariate_outlier",
            })
        all_uni_res_outlier.append(pd.DataFrame(rows))

    # ── Concat + CSV ──────────────────────────────────────────────────────────
    if len(all_uni_res_outlier) == 0:
        print("WARNING: no model converged")
    else:
        df_outlier = pd.concat(all_uni_res_outlier, ignore_index=True)
        df_outlier = df_outlier.sort_values(["feature", "variable"]).reset_index(drop=True)

        # ── Flag unstable estimates ───────────────────────────────────────────
        df_outlier["unstable"] = (
            (df_outlier["CI_low"] > df_outlier["OR"]) |
            (df_outlier["CI_high"] < df_outlier["OR"])
        )

        df_outlier.to_csv(
            os.path.join(OUT_DIR_REG, f"univariate_outlier_results_{n_clusters}clusters.csv"),
            index=False
        )
        print(f"\nSaved: univariate_outlier_results_{n_clusters}clusters.csv ({len(df_outlier)} rows)")

        n_unstable = df_outlier["unstable"].sum()
        if n_unstable > 0:
            print(f"WARNING — {n_unstable} unstable estimates (quasi-separation) flagged in CSV")

        # ── Forest plot ───────────────────────────────────────────────────────
        df_plot = df_outlier.sort_values("OR", ascending=True).reset_index(drop=True)
        n_vars  = len(df_plot)

        colors = [
            "crimson"   if (s and or_val > 1)  else
            "steelblue" if (s and or_val <= 1) else
            "lightgrey"
            for s, or_val in zip(df_plot["significant"], df_plot["OR"])
        ]

        # ── Fix unstable CI for display only ──────────────────────────────────
        ci_low_disp  = df_plot.apply(
            lambda r: r["OR"] * 0.5 if r["unstable"] else r["CI_low"], axis=1
        ).values
        ci_high_disp = df_plot.apply(
            lambda r: r["OR"] * 2.0 if r["unstable"] else r["CI_high"], axis=1
        ).values
        or_vals = df_plot["OR"].values

        fig, ax = plt.subplots(figsize=(8, max(6, n_vars * 0.35)), facecolor="white")
        ax.set_facecolor("white")

        for i in range(n_vars):
            if i % 2 == 0:
                ax.axhspan(i - 0.5, i + 0.5, color="lightgrey", alpha=0.3, zorder=0)

        for i in range(n_vars):
            ax.plot(
                [ci_low_disp[i], ci_high_disp[i]],
                [i, i],
                color=colors[i],
                linewidth=1.5,
                zorder=1,
                linestyle="--" if df_plot["unstable"].iloc[i] else "-"
            )
            marker = "^" if df_plot["unstable"].iloc[i] else "o"
            ax.scatter(
                or_vals[i],
                i,
                color=colors[i],
                s=40,
                zorder=2,
                marker=marker
            )

        ax.axvline(1, color="black", linewidth=0.8, linestyle="-")
        for x in [0.5, 2.0]:
            ax.axvline(x, color="grey", linewidth=0.6, linestyle="--", alpha=0.6)

        ax.set_xscale("log")
        ax.set_yticks(range(n_vars))
        ax.set_yticklabels(df_plot["variable"], fontsize=9)
        ax.set_xlabel("Odds Ratio (log scale)", fontsize=11)

        unstable_note = "\n(▲ = unstable estimate, quasi-separation)" if n_unstable > 0 else ""
        ax.set_title(
            f"Univariate OR — Outliers vs rest — {n_clusters} clusters\n"
            f"(crimson = sig OR>1 | steelblue = sig OR<1 | grey = ns){unstable_note}",
            fontsize=11, fontweight="bold", color="black"
        )
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

        plt.tight_layout()
        plt.savefig(
            os.path.join(OUT_DIR_REG, f"forest_plot_univariate_outliers_{n_clusters}clusters.png"),
            dpi=200, bbox_inches="tight", facecolor="white"
        )
        plt.close()
        print(f"Saved: forest_plot_univariate_outliers_{n_clusters}clusters.png")

In [ ]:
# # ==============================================================================
# # CHUNK 5 — Multivariate multinomial logistic regression
# # ==============================================================================
# for config in CLUSTERING_RUNS:
#     n_clusters     = config['n_clusters']
#     mcs            = config['mcs']
#     ms             = config['ms']
#     ref_c          = config['ref_cluster']
#     CLUSTER_LABELS = config['cluster_labels']
#     CLUSTER_ORDER  = config['cluster_order']
#     OUT_DIR_REG    = f"{BASE_DIR}/final_mcs{mcs}_ms{ms}/logistic_regression"
#
#     CSV_PATH = f"{BASE_DIR}/final_mcs{mcs}_ms{ms}/clustering_{RUN_LABEL}_{SCALER}_mcs{mcs}_ms{ms}_labeled.csv"
#     df_clust = pd.read_csv(CSV_PATH, low_memory=False)
#     df_clust['complaint_category_reg'] = df_clust['complaint_category'].replace(
#         {cat: 'Metabolic_Hematologic_Toxic' for cat in SYSTEMIC_RARE}
#     )
#     cols_needed = ALL_REG_FEATURES_WITH_TRIAGE + ['cluster', 'cluster_label', 'age']
#     df_reg = df_clust[[c for c in cols_needed if c in df_clust.columns]].copy()
#     df_reg = df_reg.dropna(subset=ALL_REG_FEATURES_WITH_TRIAGE + ['cluster'])
#     df_reg['cluster']                = df_reg['cluster'].astype(int)
#     df_reg['triage']                 = df_reg['triage'].astype(float).astype(int).astype(str)
#     df_reg['age_group']              = df_reg['age_group'].astype(str)
#     df_reg['sex']                    = df_reg['sex'].astype(str)
#     df_reg['transport_grouped']      = df_reg['transport_grouped'].astype(str)
#     df_reg['complaint_category_reg'] = df_reg['complaint_category_reg'].astype(str)
#     for col in REG_FEATURES_STATUS:
#         if col in df_reg.columns:
#             df_reg[col] = df_reg[col].astype(str)
#     df_reg['is_outlier'] = (df_reg['cluster'] == -1).astype(int)
#     df_clusters_only = df_reg[df_reg['cluster'] != -1].copy()
#
#     print("\n" + "="*60)
#     print("MULTIVARIATE REGRESSION")
#     print("="*60)
#
#     formula_parts = []
#     for var in ALL_REG_FEATURES_WITH_TRIAGE:
#         if var not in df_reg.columns:
#             continue
#         ref_val = REF_CATEGORIES.get(var, df_reg[var].mode()[0])
#         formula_parts.append(f"C({var}, Treatment('{ref_val}'))")
#
#     formula_multi = "cluster ~ " + " + ".join(formula_parts)
#     print(f"\nFormula:\n{formula_multi}\n")
#
#     # ── Fit avec fallback comme l'univarié ────────────────────────────────────────
#     model_multi = None
#     for method in ["bfgs", "lbfgs", "cg", "newton"]:
#         try:
#             model_multi = mnlogit(formula_multi, data=df_clusters_only).fit(
#                 method  = method,
#                 maxiter = 2000,
#                 gtol    = 1e-5,
#                 disp    = False,
#             )
#             if not model_multi.mle_retvals.get("converged", True):
#                 print(f"WARNING (method={method}): converged=False, résultats conservés")
#             else:
#                 print(f"OK multivariate (method={method})")
#             break
#         except Exception as e:
#             print(f"  {method} failed: {e}")
#
#     if model_multi is None:
#         print("FAILED — all methods failed")
#     else:
#         # ── Extraire résultats via format_mnlogit_results ─────────────────────────
#         df_multivariate = format_mnlogit_results(model_multi, ref_c, CLUSTER_LABELS)
#         df_multivariate["model"] = "multivariate"
#
#         df_multivariate = df_multivariate.sort_values(
#             ["cluster_vs_ref", "variable"]
#         ).reset_index(drop=True)
#
#         df_multivariate.to_csv(
#             os.path.join(OUT_DIR_REG, f"multivariate_results_{n_clusters}clusters_with_triage.csv"), index=False
#         )
#         print(f"\nSaved: multivariate_results.csv ({len(df_multivariate)} rows)")
#
#         # ── Forest plot — une seule colonne par cluster_vs_ref ────────────────────
#         clusters_vs_ref = sorted(df_multivariate["cluster_vs_ref"].unique())
#
#         ref_label = CLUSTER_LABELS.get(ref_c, f"C{ref_c}")
#
#         for clust_label in clusters_vs_ref:
#             df_plot = (
#                 df_multivariate[df_multivariate["cluster_vs_ref"] == clust_label]
#                 .sort_values("OR", ascending=True)
#                 .reset_index(drop=True)
#             )
#             n_vars = len(df_plot)
#             colors = [
#                 "crimson"   if (s and or_val > 1)  else
#                 "steelblue" if (s and or_val <= 1) else
#                 "lightgrey"
#                 for s, or_val in zip(df_plot["significant"], df_plot["OR"])
#             ]
#
#             fig, ax = plt.subplots(
#                 figsize   = (10, max(8, n_vars * 0.35)),
#                 facecolor = "white"
#             )
#             ax.set_facecolor("white")
#
#             err_low  = np.clip(np.log(df_plot["OR"]) - np.log(df_plot["CI_low"]),  0, None)
#             err_high = np.clip(np.log(df_plot["CI_high"]) - np.log(df_plot["OR"]), 0, None)
#             ax.barh(
#                 df_plot["variable"],
#                 np.log(df_plot["OR"]),
#                 xerr   = [err_low, err_high],
#                 color   = colors,
#                 alpha   = 0.8,
#                 capsize = 3,
#                 ecolor  = "grey",
#             )
#             ax.axvline(0, color="black", linewidth=0.8, linestyle="--")
#             ax.set_xlabel("log(OR) = β", fontsize=12)
#             ax.set_title(
#                 f"Multivariate OR — {clust_label}\n"
#                 f"(crimson = significant OR>1 | steelblue = significant OR<1 | grey = ns)\n"
#                 f"ref = {ref_label}",
#                 fontsize=11, fontweight="bold", color="black"
#             )
#             ax.tick_params(axis="y", labelsize=9)
#             ax.spines["top"].set_visible(False)
#             ax.spines["right"].set_visible(False)
#
#             plt.tight_layout()
#
#             # ── Nom de fichier nettoyé ────────────────────────────────────────────────
#             fname_clean = (
#                 clust_label
#                 .replace(" ", "_")
#                 .replace("—", "-")
#                 .replace("+", "plus")
#                 .replace("/", "-")
#                 .replace("(", "").replace(")", "")
#             )
#             fname = f"forest_plot_multivariate_{fname_clean}_{n_clusters}clusters_with_triage.png"
#             plt.savefig(
#                 os.path.join(OUT_DIR_REG, fname),
#                 dpi=200, bbox_inches="tight", facecolor="white"
#             )
#             plt.close()
#             print(f"Saved: {fname}")
#
#         # ── Forest plot combiné with triage ───────────────────────────────────
#         n_clusters_plot = len(clusters_vs_ref)
#         n_cols_fp = min(3, n_clusters_plot)
#         n_rows_fp = (n_clusters_plot + n_cols_fp - 1) // n_cols_fp
#
#         fig, axes = plt.subplots(
#             n_rows_fp, n_cols_fp,
#             figsize=(n_cols_fp * 10, n_rows_fp * max(8, n_vars * 0.35))
#         )
#         axes = np.array(axes).flatten()
#
#         for ax_i, clust_label in enumerate(clusters_vs_ref):
#             ax = axes[ax_i]
#             df_plot = (
#                 df_multivariate[df_multivariate["cluster_vs_ref"] == clust_label]
#                 .sort_values("OR", ascending=True)
#                 .reset_index(drop=True)
#             )
#             colors = [
#                 "crimson"   if (s and or_val > 1)  else
#                 "steelblue" if (s and or_val <= 1) else
#                 "lightgrey"
#                 for s, or_val in zip(df_plot["significant"], df_plot["OR"])
#             ]
#             err_low  = np.clip(np.log(df_plot["OR"]) - np.log(df_plot["CI_low"]),  0, None)
#             err_high = np.clip(np.log(df_plot["CI_high"]) - np.log(df_plot["OR"]), 0, None)
#             ax.barh(
#                 df_plot["variable"], np.log(df_plot["OR"]),
#                 xerr=[err_low, err_high],
#                 color=colors, alpha=0.8, capsize=3, ecolor="grey",
#             )
#             ax.axvline(0, color="black", linewidth=0.8, linestyle="--")
#             ax.set_xlabel("log(OR) = β", fontsize=10)
#             ax.set_title(clust_label, fontsize=9, fontweight="bold")
#             ax.tick_params(axis="y", labelsize=7)
#             ax.spines["top"].set_visible(False)
#             ax.spines["right"].set_visible(False)
#
#         for j in range(ax_i + 1, len(axes)):
#             axes[j].set_visible(False)
#
#         plt.suptitle(
#             f"Multivariate OR (with triage) — all clusters vs {ref_label} — {n_clusters} clusters\n"
#             f"(crimson = sig OR>1 | steelblue = sig OR<1 | grey = ns)",
#             fontsize=13, fontweight="bold", y=1.01
#         )
#         plt.tight_layout()
#         plt.savefig(
#             os.path.join(OUT_DIR_REG, f"forest_plot_multivariate_all_clusters_{n_clusters}clusters_with_triage.png"),
#             dpi=200, bbox_inches="tight"
#         )
#         plt.close()
#         print(f"Saved: forest_plot_multivariate_all_clusters_{n_clusters}clusters_with_triage.png")
#
#
#     # ── MULTIVARIATE — SANS TRIAGE ────────────────────────────────────────────
#     print("\n── Multivariate sans triage ──")
#     formula_parts_nt = []
#     for var in ALL_REG_FEATURES_NO_TRIAGE:
#         if var not in df_clusters_only.columns:
#             continue
#         ref_val = REF_CATEGORIES.get(var, df_clusters_only[var].mode()[0])
#         formula_parts_nt.append(f"C({var}, Treatment('{ref_val}'))")
#
#     formula_multi_nt = "cluster ~ " + " + ".join(formula_parts_nt)
#
#     model_multi_nt = None
#     for method in ["bfgs", "lbfgs", "cg", "newton"]:
#         try:
#             model_multi_nt = mnlogit(formula_multi_nt, data=df_clusters_only).fit(
#                 method=method, maxiter=2000, gtol=1e-5, disp=False,
#             )
#             if not model_multi_nt.mle_retvals.get("converged", True):
#                 print(f"WARNING (method={method}): converged=False")
#             else:
#                 print(f"OK multivariate no triage (method={method})")
#             break
#         except Exception as e:
#             print(f"  {method} failed: {e}")
#
#     if model_multi_nt is None:
#         print("FAILED — all methods failed")
#     else:
#         df_multi_nt = format_mnlogit_results(model_multi_nt, ref_c, CLUSTER_LABELS)
#         df_multi_nt["model"] = "multivariate_no_triage"
#         df_multi_nt = df_multi_nt.sort_values(["cluster_vs_ref", "variable"]).reset_index(drop=True)
#         df_multi_nt.to_csv(os.path.join(OUT_DIR_REG, f"multivariate_results_{n_clusters}clusters_no_triage.csv"), index=False)
#         print(f"Saved: multivariate_results_{n_clusters}clusters_no_triage.csv")
#
#         ref_label = CLUSTER_LABELS.get(ref_c, f"C{ref_c}")
#         for clust_label in sorted(df_multi_nt["cluster_vs_ref"].unique()):
#             df_plot = (
#                 df_multi_nt[df_multi_nt["cluster_vs_ref"] == clust_label]
#                 .sort_values("OR", ascending=True).reset_index(drop=True)
#             )
#             n_vars = len(df_plot)
#             colors = [
#                 "crimson"   if (s and or_val > 1)  else
#                 "steelblue" if (s and or_val <= 1) else
#                 "lightgrey"
#                 for s, or_val in zip(df_plot["significant"], df_plot["OR"])
#             ]
#             fig, ax = plt.subplots(figsize=(10, max(8, n_vars * 0.35)), facecolor="white")
#             ax.set_facecolor("white")
#             err_low  = np.clip(np.log(df_plot["OR"]) - np.log(df_plot["CI_low"]),  0, None)
#             err_high = np.clip(np.log(df_plot["CI_high"]) - np.log(df_plot["OR"]), 0, None)
#             ax.barh(
#                 df_plot["variable"],
#                 np.log(df_plot["OR"]),
#                 xerr   = [err_low, err_high],
#                 color   = colors,
#                 alpha   = 0.8,
#                 capsize = 3,
#                 ecolor  = "grey",
#             )
#             ax.axvline(0, color="black", linewidth=0.8, linestyle="--")
#             ax.set_xlabel("log(OR) = β", fontsize=12)
#             ax.set_title(
#                 f"Multivariate OR (no triage) — {clust_label}\n"
#                 f"(crimson = sig OR>1 | steelblue = sig OR<1 | grey = ns)\nref = {ref_label}",
#                 fontsize=11, fontweight="bold", color="black"
#             )
#             ax.tick_params(axis="y", labelsize=9)
#             ax.spines["top"].set_visible(False)
#             ax.spines["right"].set_visible(False)
#             plt.tight_layout()
#             fname_clean = (
#                 clust_label.replace(" ", "_").replace("—", "-")
#                 .replace("+", "plus").replace("/", "-")
#                 .replace("(", "").replace(")", "")
#             )
#             fname = f"forest_plot_multivariate_{fname_clean}_{n_clusters}clusters_no_triage.png"
#             plt.savefig(os.path.join(OUT_DIR_REG, fname), dpi=200, bbox_inches="tight", facecolor="white")
#             plt.close()
#             print(f"Saved: {fname}")
#
#         # ── Forest plot combiné no triage ─────────────────────────────────────
#         clusters_vs_ref_nt = sorted(df_multi_nt["cluster_vs_ref"].unique())
#         n_clusters_plot = len(clusters_vs_ref_nt)
#         n_cols_fp = min(3, n_clusters_plot)
#         n_rows_fp = (n_clusters_plot + n_cols_fp - 1) // n_cols_fp
#
#         fig, axes = plt.subplots(
#             n_rows_fp, n_cols_fp,
#             figsize=(n_cols_fp * 10, n_rows_fp * max(8, n_vars * 0.35))
#         )
#         axes = np.array(axes).flatten()
#
#         for ax_i, clust_label in enumerate(clusters_vs_ref_nt):
#             ax = axes[ax_i]
#             df_plot = (
#                 df_multi_nt[df_multi_nt["cluster_vs_ref"] == clust_label]
#                 .sort_values("OR", ascending=True)
#                 .reset_index(drop=True)
#             )
#             colors = [
#                 "crimson"   if (s and or_val > 1)  else
#                 "steelblue" if (s and or_val <= 1) else
#                 "lightgrey"
#                 for s, or_val in zip(df_plot["significant"], df_plot["OR"])
#             ]
#             err_low  = np.clip(np.log(df_plot["OR"]) - np.log(df_plot["CI_low"]),  0, None)
#             err_high = np.clip(np.log(df_plot["CI_high"]) - np.log(df_plot["OR"]), 0, None)
#             ax.barh(
#                 df_plot["variable"], np.log(df_plot["OR"]),
#                 xerr=[err_low, err_high],
#                 color=colors, alpha=0.8, capsize=3, ecolor="grey",
#             )
#             ax.axvline(0, color="black", linewidth=0.8, linestyle="--")
#             ax.set_xlabel("log(OR) = β", fontsize=10)
#             ax.set_title(clust_label, fontsize=9, fontweight="bold")
#             ax.tick_params(axis="y", labelsize=7)
#             ax.spines["top"].set_visible(False)
#             ax.spines["right"].set_visible(False)
#
#         for j in range(ax_i + 1, len(axes)):
#             axes[j].set_visible(False)
#
#         plt.suptitle(
#             f"Multivariate OR (no triage) — all clusters vs {ref_label} — {n_clusters} clusters\n"
#             f"(crimson = sig OR>1 | steelblue = sig OR<1 | grey = ns)",
#             fontsize=13, fontweight="bold", y=1.01
#         )
#         plt.tight_layout()
#         plt.savefig(
#             os.path.join(OUT_DIR_REG, f"forest_plot_multivariate_all_clusters_{n_clusters}clusters_no_triage.png"),
#             dpi=200, bbox_inches="tight"
#         )
#         plt.close()
#         print(f"Saved: forest_plot_multivariate_all_clusters_{n_clusters}clusters_no_triage.png")
#
#
#     # ── MULTIVARIATE OUTLIERS — with triage ───────────────────────────────────
#     print("\n── Multivariate outliers avec triage ──")
#     formula_parts_out = []
#     for var in ALL_REG_FEATURES_WITH_TRIAGE:
#         if var not in df_reg.columns:
#             continue
#         ref_val = REF_CATEGORIES.get(var, df_reg[var].mode()[0])
#         formula_parts_out.append(f"C({var}, Treatment('{ref_val}'))")
#
#     formula_out = "is_outlier ~ " + " + ".join(formula_parts_out)
#
#     model_out = None
#     for method in ["bfgs", "lbfgs", "cg", "newton"]:
#         try:
#             model_out = smf.logit(formula_out, data=df_reg).fit(
#                 method=method, maxiter=2000, gtol=1e-5, disp=False,
#             )
#             if not model_out.mle_retvals.get("converged", True):
#                 print(f"WARNING (method={method}): converged=False")
#             else:
#                 print(f"OK multivariate outliers with triage (method={method})")
#             break
#         except Exception as e:
#             print(f"  {method} failed: {e}")
#
#     if model_out is None:
#         print("FAILED — all methods failed")
#     else:
#         params = model_out.params
#         conf   = model_out.conf_int()
#         pvals  = model_out.pvalues
#         conf.columns = ["CI_low", "CI_high"]
#         rows = []
#         for v in params.index:
#             if v == "Intercept":
#                 continue
#             clean_var = re.sub(r"C\((.+?),\s*Treatment\(.+?\)\)\[T\.(.+?)\]", r"\1 = \2", v)
#             rows.append({
#                 "variable"   : clean_var,
#                 "OR"         : round(np.exp(params[v]), 3),
#                 "CI_low"     : round(np.exp(conf.loc[v, "CI_low"]), 3),
#                 "CI_high"    : round(np.exp(conf.loc[v, "CI_high"]), 3),
#                 "p_value"    : round(pvals[v], 4),
#                 "significant": pvals[v] < 0.05,
#             })
#         df_out_multi_wt = pd.DataFrame(rows)
#         df_out_multi_wt.to_csv(os.path.join(OUT_DIR_REG, f"multivariate_outlier_results_{n_clusters}clusters_with_triage.csv"), index=False)
#
#         df_plot = df_out_multi_wt.sort_values("OR", ascending=True).reset_index(drop=True)
#         n_vars  = len(df_plot)
#         colors  = [
#             "crimson"   if (s and or_val > 1)  else
#             "steelblue" if (s and or_val <= 1) else
#             "lightgrey"
#             for s, or_val in zip(df_plot["significant"], df_plot["OR"])
#         ]
#         err_low  = np.clip(np.log(df_plot["OR"]) - np.log(df_plot["CI_low"]),  0, None)
#         err_high = np.clip(np.log(df_plot["CI_high"]) - np.log(df_plot["OR"]), 0, None)
#         fig, ax = plt.subplots(figsize=(10, max(8, n_vars * 0.35)))
#         ax.barh(df_plot["variable"], np.log(df_plot["OR"]),
#                 xerr=[err_low, err_high], color=colors, alpha=0.8, capsize=3, ecolor="grey")
#         ax.axvline(0, color="black", linewidth=0.8, linestyle="--")
#         ax.set_xlabel("log(OR) = β", fontsize=12)
#         ax.set_title(
#             f"Multivariate OR (with triage) — Outliers vs rest — {n_clusters} clusters\n"
#             f"(crimson = sig OR>1 | steelblue = sig OR<1 | grey = ns)",
#             fontsize=11, fontweight="bold"
#         )
#         ax.tick_params(axis="y", labelsize=9)
#         ax.spines["top"].set_visible(False)
#         ax.spines["right"].set_visible(False)
#         plt.tight_layout()
#         plt.savefig(os.path.join(OUT_DIR_REG, f"forest_plot_multivariate_outliers_{n_clusters}clusters_with_triage.png"),
#                     dpi=200, bbox_inches="tight")
#         plt.close()
#         print(f"Saved: forest_plot_multivariate_outliers_{n_clusters}clusters_with_triage.png")
#
#     # ── MULTIVARIATE OUTLIERS — no triage ─────────────────────────────────────
#     print("\n── Multivariate outliers sans triage ──")
#     formula_parts_out_nt = []
#     for var in ALL_REG_FEATURES_NO_TRIAGE:
#         if var not in df_reg.columns:
#             continue
#         ref_val = REF_CATEGORIES.get(var, df_reg[var].mode()[0])
#         formula_parts_out_nt.append(f"C({var}, Treatment('{ref_val}'))")
#
#     formula_out_nt = "is_outlier ~ " + " + ".join(formula_parts_out_nt)
#
#     model_out_nt = None
#     for method in ["bfgs", "lbfgs", "cg", "newton"]:
#         try:
#             model_out_nt = smf.logit(formula_out_nt, data=df_reg).fit(
#                 method=method, maxiter=2000, gtol=1e-5, disp=False,
#             )
#             if not model_out_nt.mle_retvals.get("converged", True):
#                 print(f"WARNING (method={method}): converged=False")
#             else:
#                 print(f"OK multivariate outliers no triage (method={method})")
#             break
#         except Exception as e:
#             print(f"  {method} failed: {e}")
#
#     if model_out_nt is None:
#         print("FAILED — all methods failed")
#     else:
#         params = model_out_nt.params
#         conf   = model_out_nt.conf_int()
#         pvals  = model_out_nt.pvalues
#         conf.columns = ["CI_low", "CI_high"]
#         rows = []
#         for v in params.index:
#             if v == "Intercept":
#                 continue
#             clean_var = re.sub(r"C\((.+?),\s*Treatment\(.+?\)\)\[T\.(.+?)\]", r"\1 = \2", v)
#             rows.append({
#                 "variable"   : clean_var,
#                 "OR"         : round(np.exp(params[v]), 3),
#                 "CI_low"     : round(np.exp(conf.loc[v, "CI_low"]), 3),
#                 "CI_high"    : round(np.exp(conf.loc[v, "CI_high"]), 3),
#                 "p_value"    : round(pvals[v], 4),
#                 "significant": pvals[v] < 0.05,
#             })
#         df_out_multi_nt = pd.DataFrame(rows)
#         df_out_multi_nt.to_csv(os.path.join(OUT_DIR_REG, f"multivariate_outlier_results_{n_clusters}clusters_no_triage.csv"), index=False)
#
#         df_plot = df_out_multi_nt.sort_values("OR", ascending=True).reset_index(drop=True)
#         n_vars  = len(df_plot)
#         colors  = [
#             "crimson"   if (s and or_val > 1)  else
#             "steelblue" if (s and or_val <= 1) else
#             "lightgrey"
#             for s, or_val in zip(df_plot["significant"], df_plot["OR"])
#         ]
#         err_low  = np.clip(np.log(df_plot["OR"]) - np.log(df_plot["CI_low"]),  0, None)
#         err_high = np.clip(np.log(df_plot["CI_high"]) - np.log(df_plot["OR"]), 0, None)
#         fig, ax = plt.subplots(figsize=(10, max(8, n_vars * 0.35)))
#         ax.barh(df_plot["variable"], np.log(df_plot["OR"]),
#                 xerr=[err_low, err_high], color=colors, alpha=0.8, capsize=3, ecolor="grey")
#         ax.axvline(0, color="black", linewidth=0.8, linestyle="--")
#         ax.set_xlabel("log(OR) = β", fontsize=12)
#         ax.set_title(
#             f"Multivariate OR (no triage) — Outliers vs rest — {n_clusters} clusters\n"
#             f"(crimson = sig OR>1 | steelblue = sig OR<1 | grey = ns)",
#             fontsize=11, fontweight="bold"
#         )
#         ax.tick_params(axis="y", labelsize=9)
#         ax.spines["top"].set_visible(False)
#         ax.spines["right"].set_visible(False)
#         plt.tight_layout()
#         plt.savefig(os.path.join(OUT_DIR_REG, f"forest_plot_multivariate_outliers_{n_clusters}clusters_no_triage.png"),
#                     dpi=200, bbox_inches="tight")
#         plt.close()
#         print(f"Saved: forest_plot_multivariate_outliers_{n_clusters}clusters_no_triage.png")

In [ ]:
# ==============================================================================
# CHUNK 5 — Multivariate multinomial logistic regression
# ==============================================================================

import statsmodels.formula.api as smf

for config in CLUSTERING_RUNS:
    n_clusters     = config['n_clusters']
    mcs            = config['mcs']
    ms             = config['ms']
    ref_c          = config['ref_cluster']
    CLUSTER_LABELS = config['cluster_labels']
    CLUSTER_ORDER  = config['cluster_order']
    OUT_DIR_REG    = f"{BASE_DIR}/final_mcs{mcs}_ms{ms}/logistic_regression"
    os.makedirs(OUT_DIR_REG, exist_ok=True)

    # ── Load + remaps ─────────────────────────────────────────────────────────
    CSV_PATH = f"{BASE_DIR}/final_mcs{mcs}_ms{ms}/clustering_{RUN_LABEL}_{SCALER}_mcs{mcs}_ms{ms}_labeled.csv"
    df_clust = pd.read_csv(CSV_PATH, low_memory=False)

    for col, mapping in VITAL_SIGN_REMAP.items():
        if col in df_clust.columns:
            df_clust[col] = df_clust[col].map(mapping)

    df_clust['triage_grouped'] = (
        df_clust['triage'].astype(float).astype(int)
        .map(TRIAGE_GROUP_MAP).astype(str)
    )
    df_clust['complaint_category_reg'] = df_clust['complaint_category'].replace(
        {cat: 'Other' for cat in SYSTEMIC_RARE}
    )
    df_clust['cluster_label'] = df_clust['cluster'].map(CLUSTER_LABELS)
    df_clust['cluster_label'] = pd.Categorical(
        df_clust['cluster_label'], categories=CLUSTER_ORDER, ordered=True
    )

    # ── Build df_reg ──────────────────────────────────────────────────────────
    cols_needed = list(set(
        ALL_REG_FEATURES_WITH_TRIAGE + ALL_REG_FEATURES_NO_TRIAGE +
        ALL_REG_FEATURES_WITH_TRIAGE_MULTI + ALL_REG_FEATURES_NO_TRIAGE_MULTI +
        ['cluster', 'cluster_label', 'age']
    ))
    df_reg = df_clust[[c for c in cols_needed if c in df_clust.columns]].copy()
    df_reg = df_reg.dropna(subset=ALL_REG_FEATURES_WITH_TRIAGE_MULTI + ['cluster'])
    df_reg['cluster']                = df_reg['cluster'].astype(int)
    df_reg['triage_grouped']         = df_reg['triage_grouped'].astype(str)
    df_reg['age_group']              = df_reg['age_group'].astype(str)
    df_reg['sex']                    = df_reg['sex'].astype(str)
    df_reg['transport_grouped']      = df_reg['transport_grouped'].astype(str)
    df_reg['complaint_category_reg'] = df_reg['complaint_category_reg'].astype(str)
    for col in REG_FEATURES_STATUS_MULTI:
        if col in df_reg.columns:
            df_reg[col] = df_reg[col].astype(str)
    df_reg['is_outlier']  = (df_reg['cluster'] == -1).astype(int)
    df_clusters_only      = df_reg[df_reg['cluster'] != -1].copy()

    ref_label = CLUSTER_LABELS.get(ref_c, f"C{ref_c}")

    print("\n" + "="*60)
    print(f"MULTIVARIATE REGRESSION — {n_clusters} clusters")
    print("="*60)
    print(f"Regression dataset: {len(df_reg):,} patients")

    # ── Helper — flag unstable estimates ──────────────────────────────────────
    def flag_unstable(df):
        df = df.copy()
        df["unstable"] = (
            (df["CI_low"] > df["OR"]) | (df["CI_high"] < df["OR"])
        )
        return df

    # ── Helper — forest plot points + CI style with unstable flag ─────────────
    def make_forest_plot(df_input, title, fname, out_dir):
        df_plot = flag_unstable(df_input).sort_values("OR", ascending=True).reset_index(drop=True)
        n_vars  = len(df_plot)

        colors = [
            "crimson"   if (s and or_val > 1)  else
            "steelblue" if (s and or_val <= 1) else
            "lightgrey"
            for s, or_val in zip(df_plot["significant"], df_plot["OR"])
        ]

        ci_low_disp  = df_plot.apply(
            lambda r: r["OR"] * 0.5 if r["unstable"] else r["CI_low"], axis=1
        ).values
        ci_high_disp = df_plot.apply(
            lambda r: r["OR"] * 2.0 if r["unstable"] else r["CI_high"], axis=1
        ).values
        or_vals = df_plot["OR"].values

        fig, ax = plt.subplots(figsize=(8, max(6, n_vars * 0.35)), facecolor="white")
        ax.set_facecolor("white")

        for i in range(n_vars):
            if i % 2 == 0:
                ax.axhspan(i - 0.5, i + 0.5, color="lightgrey", alpha=0.3, zorder=0)

        for i in range(n_vars):
            ax.plot(
                [ci_low_disp[i], ci_high_disp[i]],
                [i, i],
                color=colors[i],
                linewidth=1.5,
                zorder=1,
                linestyle="--" if df_plot["unstable"].iloc[i] else "-"
            )
            marker = "^" if df_plot["unstable"].iloc[i] else "o"
            ax.scatter(or_vals[i], i, color=colors[i], s=40, zorder=2, marker=marker)

        ax.axvline(1, color="black", linewidth=0.8, linestyle="-")
        for x in [0.5, 2.0]:
            ax.axvline(x, color="grey", linewidth=0.6, linestyle="--", alpha=0.6)

        ax.set_xscale("log")
        ax.set_yticks(range(n_vars))
        ax.set_yticklabels(df_plot["variable"], fontsize=9)
        ax.set_xlabel("Odds Ratio (log scale)", fontsize=11)

        n_unstable = df_plot["unstable"].sum()
        unstable_note = "\n(▲ = unstable estimate, quasi-separation)" if n_unstable > 0 else ""
        ax.set_title(f"{title}{unstable_note}", fontsize=11, fontweight="bold", color="black")
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        plt.tight_layout()
        plt.savefig(os.path.join(out_dir, fname), dpi=200, bbox_inches="tight", facecolor="white")
        plt.close()
        print(f"Saved: {fname}")

    def make_combined_forest_plot(df_results, clusters_vs_ref, title, fname, out_dir):
        n_clusters_plot = len(clusters_vs_ref)
        n_cols_fp = min(3, n_clusters_plot)
        n_rows_fp = (n_clusters_plot + n_cols_fp - 1) // n_cols_fp

        fig, axes = plt.subplots(
            n_rows_fp, n_cols_fp,
            figsize=(n_cols_fp * 8, n_rows_fp * 6)
        )
        axes = np.array(axes).flatten()

        for ax_i, clust_label in enumerate(clusters_vs_ref):
            ax = axes[ax_i]
            df_plot = flag_unstable(
                df_results[df_results["cluster_vs_ref"] == clust_label]
                .sort_values("OR", ascending=True)
                .reset_index(drop=True)
            )
            n_vars_i = len(df_plot)
            colors = [
                "crimson"   if (s and or_val > 1)  else
                "steelblue" if (s and or_val <= 1) else
                "lightgrey"
                for s, or_val in zip(df_plot["significant"], df_plot["OR"])
            ]

            ci_low_disp  = df_plot.apply(
                lambda r: r["OR"] * 0.5 if r["unstable"] else r["CI_low"], axis=1
            ).values
            ci_high_disp = df_plot.apply(
                lambda r: r["OR"] * 2.0 if r["unstable"] else r["CI_high"], axis=1
            ).values
            or_vals = df_plot["OR"].values

            for i in range(n_vars_i):
                if i % 2 == 0:
                    ax.axhspan(i - 0.5, i + 0.5, color="lightgrey", alpha=0.3, zorder=0)

            for i in range(n_vars_i):
                ax.plot(
                    [ci_low_disp[i], ci_high_disp[i]],
                    [i, i],
                    color=colors[i],
                    linewidth=1.5,
                    zorder=1,
                    linestyle="--" if df_plot["unstable"].iloc[i] else "-"
                )
                marker = "^" if df_plot["unstable"].iloc[i] else "o"
                ax.scatter(or_vals[i], i, color=colors[i], s=30, zorder=2, marker=marker)

            ax.axvline(1, color="black", linewidth=0.8, linestyle="-")
            for x in [0.5, 2.0]:
                ax.axvline(x, color="grey", linewidth=0.6, linestyle="--", alpha=0.6)

            ax.set_xscale("log")
            ax.set_yticks(range(n_vars_i))
            ax.set_yticklabels(df_plot["variable"], fontsize=7)
            ax.set_xlabel("OR (log scale)", fontsize=9)
            ax.set_title(clust_label, fontsize=8, fontweight="bold")
            ax.spines["top"].set_visible(False)
            ax.spines["right"].set_visible(False)

        for j in range(ax_i + 1, len(axes)):
            axes[j].set_visible(False)

        plt.suptitle(
            f"{title}\n(▲ = unstable estimate, quasi-separation)",
            fontsize=13, fontweight="bold", y=1.01
        )
        plt.tight_layout()
        plt.savefig(os.path.join(out_dir, fname), dpi=200, bbox_inches="tight")
        plt.close()
        print(f"Saved: {fname}")

    # ── Helper — extract binary logistic results ──────────────────────────────
    def extract_binary_logit_results(model):
        params = model.params
        conf   = model.conf_int()
        pvals  = model.pvalues
        conf.columns = ["CI_low", "CI_high"]
        rows = []
        for v in params.index:
            if v == "Intercept":
                continue
            clean_var = re.sub(r"C\((.+?),\s*Treatment\(.+?\)\)\[T\.(.+?)\]", r"\1 = \2", v)
            rows.append({
                "variable"   : clean_var,
                "OR"         : round(np.exp(params[v]), 3),
                "CI_low"     : round(np.exp(conf.loc[v, "CI_low"]), 3),
                "CI_high"    : round(np.exp(conf.loc[v, "CI_high"]), 3),
                "p_value"    : round(pvals[v], 4),
                "significant": pvals[v] < 0.05,
            })
        return pd.DataFrame(rows)

    # ── MULTIVARIATE WITH TRIAGE ──────────────────────────────────────────────
    print("\n── Multivariate with triage ──")
    formula_parts = []
    for var in ALL_REG_FEATURES_WITH_TRIAGE_MULTI:
        if var not in df_clusters_only.columns:
            continue
        ref_val = REF_CATEGORIES.get(var, df_clusters_only[var].mode()[0])
        formula_parts.append(f"C({var}, Treatment('{ref_val}'))")

    formula_multi = "cluster ~ " + " + ".join(formula_parts)
    print(f"\nFormula:\n{formula_multi}\n")

    model_multi = None
    for method in ["bfgs", "lbfgs", "cg", "newton"]:
        try:
            model_multi = mnlogit(formula_multi, data=df_clusters_only).fit(
                method=method, maxiter=2000, gtol=1e-5, disp=False,
            )
            if not model_multi.mle_retvals.get("converged", True):
                print(f"WARNING (method={method}): converged=False")
            else:
                print(f"OK multivariate with triage (method={method})")
            break
        except Exception as e:
            print(f"  {method} failed: {e}")

    if model_multi is None:
        print("FAILED — all methods failed")
    else:
        df_multivariate = format_mnlogit_results(model_multi, ref_c, CLUSTER_LABELS)
        df_multivariate["model"] = "multivariate_with_triage"
        df_multivariate = df_multivariate.sort_values(["cluster_vs_ref", "variable"]).reset_index(drop=True)
        df_multivariate.to_csv(
            os.path.join(OUT_DIR_REG, f"multivariate_results_{n_clusters}clusters_with_triage.csv"),
            index=False
        )
        print(f"Saved: multivariate_results_{n_clusters}clusters_with_triage.csv ({len(df_multivariate)} rows)")

        clusters_vs_ref = sorted(df_multivariate["cluster_vs_ref"].unique())

        for clust_label in clusters_vs_ref:
            df_plot = (
                df_multivariate[df_multivariate["cluster_vs_ref"] == clust_label]
                .sort_values("OR", ascending=True).reset_index(drop=True)
            )
            fname_clean = (
                clust_label.replace(" ", "_").replace("—", "-")
                .replace("+", "plus").replace("/", "-")
                .replace("(", "").replace(")", "")
            )
            make_forest_plot(
                df_plot,
                title=f"Multivariate OR (with triage) — {clust_label}\nref = {ref_label}",
                fname=f"forest_plot_multivariate_{fname_clean}_{n_clusters}clusters_with_triage.png",
                out_dir=OUT_DIR_REG
            )

        make_combined_forest_plot(
            df_multivariate,
            clusters_vs_ref,
            title=f"Multivariate OR (with triage) — all clusters vs {ref_label} — {n_clusters} clusters\n(crimson = sig OR>1 | steelblue = sig OR<1 | grey = ns)",
            fname=f"forest_plot_multivariate_all_clusters_{n_clusters}clusters_with_triage.png",
            out_dir=OUT_DIR_REG
        )

    # ── MULTIVARIATE NO TRIAGE ────────────────────────────────────────────────
    print("\n── Multivariate without triage ──")
    formula_parts_nt = []
    for var in ALL_REG_FEATURES_NO_TRIAGE_MULTI:
        if var not in df_clusters_only.columns:
            continue
        ref_val = REF_CATEGORIES.get(var, df_clusters_only[var].mode()[0])
        formula_parts_nt.append(f"C({var}, Treatment('{ref_val}'))")

    formula_multi_nt = "cluster ~ " + " + ".join(formula_parts_nt)

    model_multi_nt = None
    for method in ["bfgs", "lbfgs", "cg", "newton"]:
        try:
            model_multi_nt = mnlogit(formula_multi_nt, data=df_clusters_only).fit(
                method=method, maxiter=2000, gtol=1e-5, disp=False,
            )
            if not model_multi_nt.mle_retvals.get("converged", True):
                print(f"WARNING (method={method}): converged=False")
            else:
                print(f"OK multivariate no triage (method={method})")
            break
        except Exception as e:
            print(f"  {method} failed: {e}")

    if model_multi_nt is None:
        print("FAILED — all methods failed")
    else:
        df_multi_nt = format_mnlogit_results(model_multi_nt, ref_c, CLUSTER_LABELS)
        df_multi_nt["model"] = "multivariate_no_triage"
        df_multi_nt = df_multi_nt.sort_values(["cluster_vs_ref", "variable"]).reset_index(drop=True)
        df_multi_nt.to_csv(
            os.path.join(OUT_DIR_REG, f"multivariate_results_{n_clusters}clusters_no_triage.csv"),
            index=False
        )
        print(f"Saved: multivariate_results_{n_clusters}clusters_no_triage.csv ({len(df_multi_nt)} rows)")

        clusters_vs_ref_nt = sorted(df_multi_nt["cluster_vs_ref"].unique())

        for clust_label in clusters_vs_ref_nt:
            df_plot = (
                df_multi_nt[df_multi_nt["cluster_vs_ref"] == clust_label]
                .sort_values("OR", ascending=True).reset_index(drop=True)
            )
            fname_clean = (
                clust_label.replace(" ", "_").replace("—", "-")
                .replace("+", "plus").replace("/", "-")
                .replace("(", "").replace(")", "")
            )
            make_forest_plot(
                df_plot,
                title=f"Multivariate OR (no triage) — {clust_label}\nref = {ref_label}",
                fname=f"forest_plot_multivariate_{fname_clean}_{n_clusters}clusters_no_triage.png",
                out_dir=OUT_DIR_REG
            )

        make_combined_forest_plot(
            df_multi_nt,
            clusters_vs_ref_nt,
            title=f"Multivariate OR (no triage) — all clusters vs {ref_label} — {n_clusters} clusters\n(crimson = sig OR>1 | steelblue = sig OR<1 | grey = ns)",
            fname=f"forest_plot_multivariate_all_clusters_{n_clusters}clusters_no_triage.png",
            out_dir=OUT_DIR_REG
        )

        # ── LRT + McFadden R² comparison ──────────────────────────────────────
        if model_multi is not None:
            from scipy.stats import chi2
            ll_wt   = model_multi.llf
            ll_nt   = model_multi_nt.llf
            lr_stat = 2 * (ll_wt - ll_nt)
            df_diff = model_multi.df_model - model_multi_nt.df_model
            p_lrt   = chi2.sf(lr_stat, df_diff)
            r2_wt   = 1 - ll_wt / model_multi.llnull
            r2_nt   = 1 - ll_nt / model_multi_nt.llnull

            print(f"\n── Model comparison ──────────────────────────────────────")
            print(f"McFadden R² with triage    : {r2_wt:.4f}")
            print(f"McFadden R² without triage : {r2_nt:.4f}")
            print(f"LRT statistic              : {lr_stat:.2f}")
            print(f"df difference              : {df_diff}")
            print(f"p-value                    : {p_lrt:.4e}")

    # ── MULTIVARIATE OUTLIERS — WITH TRIAGE ───────────────────────────────────
    print("\n── Multivariate outliers with triage ──")
    formula_parts_out = []
    for var in ALL_REG_FEATURES_WITH_TRIAGE_MULTI:
        if var not in df_reg.columns:
            continue
        ref_val = REF_CATEGORIES.get(var, df_reg[var].mode()[0])
        formula_parts_out.append(f"C({var}, Treatment('{ref_val}'))")

    formula_out = "is_outlier ~ " + " + ".join(formula_parts_out)

    model_out = None
    for method in ["bfgs", "lbfgs", "cg", "newton"]:
        try:
            model_out = smf.logit(formula_out, data=df_reg).fit(
                method=method, maxiter=2000, gtol=1e-5, disp=False,
            )
            if not model_out.mle_retvals.get("converged", True):
                print(f"WARNING (method={method}): converged=False")
            else:
                print(f"OK multivariate outliers with triage (method={method})")
            break
        except Exception as e:
            print(f"  {method} failed: {e}")

    if model_out is None:
        print("FAILED — all methods failed")
    else:
        df_out_multi_wt = extract_binary_logit_results(model_out)
        df_out_multi_wt.to_csv(
            os.path.join(OUT_DIR_REG, f"multivariate_outlier_results_{n_clusters}clusters_with_triage.csv"),
            index=False
        )
        make_forest_plot(
            df_out_multi_wt,
            title=f"Multivariate OR (with triage) — Outliers vs rest — {n_clusters} clusters\n(crimson = sig OR>1 | steelblue = sig OR<1 | grey = ns)",
            fname=f"forest_plot_multivariate_outliers_{n_clusters}clusters_with_triage.png",
            out_dir=OUT_DIR_REG
        )

    # ── MULTIVARIATE OUTLIERS — NO TRIAGE ─────────────────────────────────────
    print("\n── Multivariate outliers without triage ──")
    formula_parts_out_nt = []
    for var in ALL_REG_FEATURES_NO_TRIAGE_MULTI:
        if var not in df_reg.columns:
            continue
        ref_val = REF_CATEGORIES.get(var, df_reg[var].mode()[0])
        formula_parts_out_nt.append(f"C({var}, Treatment('{ref_val}'))")

    formula_out_nt = "is_outlier ~ " + " + ".join(formula_parts_out_nt)

    model_out_nt = None
    for method in ["bfgs", "lbfgs", "cg", "newton"]:
        try:
            model_out_nt = smf.logit(formula_out_nt, data=df_reg).fit(
                method=method, maxiter=2000, gtol=1e-5, disp=False,
            )
            if not model_out_nt.mle_retvals.get("converged", True):
                print(f"WARNING (method={method}): converged=False")
            else:
                print(f"OK multivariate outliers no triage (method={method})")
            break
        except Exception as e:
            print(f"  {method} failed: {e}")

    if model_out_nt is None:
        print("FAILED — all methods failed")
    else:
        df_out_multi_nt = extract_binary_logit_results(model_out_nt)
        df_out_multi_nt.to_csv(
            os.path.join(OUT_DIR_REG, f"multivariate_outlier_results_{n_clusters}clusters_no_triage.csv"),
            index=False
        )
        make_forest_plot(
            df_out_multi_nt,
            title=f"Multivariate OR (no triage) — Outliers vs rest — {n_clusters} clusters\n(crimson = sig OR>1 | steelblue = sig OR<1 | grey = ns)",
            fname=f"forest_plot_multivariate_outliers_{n_clusters}clusters_no_triage.png",
            out_dir=OUT_DIR_REG
        )

# resultats univarié dit que urine dospistick et glycemie trop instable et pas vraiment asoscié avec les clsuters donc on les vire

In [ ]:
# ==============================================================================
# CHUNK 5bis — Ridge L2 multinomial logistic regression
# C selected by 5-fold cross-validation
# ==============================================================================

from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
import pandas as pd
import numpy as np
import os
import matplotlib
import matplotlib.pyplot as plt

for config in CLUSTERING_RUNS:
    n_clusters     = config['n_clusters']
    mcs            = config['mcs']
    ms             = config['ms']
    ref_c          = config['ref_cluster']
    CLUSTER_LABELS = config['cluster_labels']
    CLUSTER_ORDER  = config['cluster_order']
    OUT_DIR_REG    = f"{BASE_DIR}/final_mcs{mcs}_ms{ms}/logistic_regression"
    os.makedirs(OUT_DIR_REG, exist_ok=True)

    # ── Load + remaps ─────────────────────────────────────────────────────────
    CSV_PATH = f"{BASE_DIR}/final_mcs{mcs}_ms{ms}/clustering_{RUN_LABEL}_{SCALER}_mcs{mcs}_ms{ms}_labeled.csv"
    df_clust = pd.read_csv(CSV_PATH, low_memory=False)

    for col, mapping in VITAL_SIGN_REMAP.items():
        if col in df_clust.columns:
            df_clust[col] = df_clust[col].map(mapping)

    df_clust['triage_grouped'] = (
        df_clust['triage'].astype(float).astype(int)
        .map(TRIAGE_GROUP_MAP).astype(str)
    )
    df_clust['complaint_category_reg'] = df_clust['complaint_category'].replace(
        {cat: 'Other' for cat in SYSTEMIC_RARE}
    )

    # ── Feature lists pour Ridge ───────────────────────────────────────────────
    RIDGE_FEATURES_CATEG_WITH_TRIAGE = [
        "sex", "age_group", "complaint_category_reg",
        "triage_grouped", "transport_grouped"
    ]
    RIDGE_FEATURES_CATEG_NO_TRIAGE = [
        "sex", "age_group", "complaint_category_reg", "transport_grouped"
    ]
    RIDGE_FEATURES_STATUS = [
        "bp_status", "hr_status", "temp_status", "sat_status",
        "rr_status", "o2_flow_status", "gcs_status", "pain_status",
        "breathalyzer_status", "urine_dipstick_clean_status", "cap_blood_sugar_status", "pupils_status", "anisocoria_status"

    ]
    ALL_RIDGE_WITH_TRIAGE = RIDGE_FEATURES_CATEG_WITH_TRIAGE + RIDGE_FEATURES_STATUS
    ALL_RIDGE_NO_TRIAGE   = RIDGE_FEATURES_CATEG_NO_TRIAGE   + RIDGE_FEATURES_STATUS

    # ── Build df_reg ──────────────────────────────────────────────────────────
    cols_needed = list(set(ALL_RIDGE_WITH_TRIAGE + ['cluster']))
    df_reg = df_clust[[c for c in cols_needed if c in df_clust.columns]].copy()
    # ── Identify sources of NaN ───────────────────────────────────────────────────
    print(f"\nN avant dropna : {len(df_reg)}")
    for col in ALL_RIDGE_WITH_TRIAGE:
        if col not in df_reg.columns:
            print(f"  MISSING: {col}")
            continue
        n_nan = df_reg[col].isna().sum()
        if n_nan > 0:
            print(f"  {col} : {n_nan} NaN")
            # Montrer les valeurs originales qui ont généré ces NaN
            if col in df_clust.columns:
                # Récupérer les valeurs avant remap
                original_vals = df_clust.loc[df_reg[df_reg[col].isna()].index, col]
                print(f"    Valeurs originales : {original_vals.value_counts().to_dict()}")


    df_reg = df_reg.dropna(subset=ALL_RIDGE_WITH_TRIAGE + ['cluster'])
    print(f"N après dropna : {df_reg.dropna(subset=ALL_RIDGE_WITH_TRIAGE).shape[0]}")
    df_reg['cluster'] = df_reg['cluster'].astype(int)
    df_clusters_only  = df_reg[df_reg['cluster'] != -1].copy()

    ref_label = CLUSTER_LABELS.get(ref_c, f"C{ref_c}")

    print("\n" + "="*60)
    print(f"RIDGE L2 MULTINOMIAL — {n_clusters} clusters")
    print("="*60)
    print(f"N = {len(df_clusters_only):,} patients")

    # ── Helper — one-hot encode avec références explicites ────────────────────
    def encode_with_ref(df, features, ref_categories):
        dummies_list = []
        feature_map  = {}
        for feat in features:
            if feat not in df.columns:
                continue
            ref     = ref_categories.get(feat, df[feat].mode()[0])
            dummies = pd.get_dummies(df[feat], prefix=feat, drop_first=False)
            ref_col = f"{feat}_{ref}"
            if ref_col in dummies.columns:
                dummies = dummies.drop(columns=[ref_col])
            for col in dummies.columns:
                modality = col.replace(f"{feat}_", "")
                feature_map[col] = f"{feat} = {modality}"
            dummies_list.append(dummies)
        X = pd.concat(dummies_list, axis=1).astype(float)
        return X, feature_map

    # ── Helper — sélection de C par cross-validation ──────────────────────────
    def select_C_by_cv(X, y, Cs=None, n_splits=5, random_state=42):
        if Cs is None:
            Cs = [0.01, 0.1, 0.5, 1.0, 5.0, 10.0]
        cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
        clf_cv = LogisticRegressionCV(
            Cs           = Cs,
            cv           = cv,
            penalty      = 'l2',
            solver       = 'lbfgs',
            max_iter     = 2000,
            scoring      = 'neg_log_loss',
            random_state = random_state,
            n_jobs       = -1,
        )
        clf_cv.fit(X, y)
        # En multinomial, clf_cv.C_ est un array de longueur n_classes
        # On prend la médiane pour avoir un seul C
        C_optimal = float(np.median(clf_cv.C_))
        print(f"  C values per class : {clf_cv.C_}")
        print(f"  C optimal (median) : {C_optimal}")
        return C_optimal, clf_cv.C_

    # ── Helper — OR + CI bootstrap ────────────────────────────────────────────
    def ridge_results_with_ci(
        X, y, classes, ref_class,
        C=1.0, n_bootstrap=200, random_state=42
    ):
        rng = np.random.default_rng(random_state)
        n   = len(X)

        # Point estimate
        clf = LogisticRegression(
            penalty      = 'l2',
            C            = C,
            solver       = 'lbfgs',
            max_iter     = 2000,
            random_state = 42
        )
        clf.fit(X, y)

        class_to_idx = {c: i for i, c in enumerate(clf.classes_)}
        ref_idx      = class_to_idx[ref_class]

        # Bootstrap
        boot_coefs = []
        for b in range(n_bootstrap):
            idx   = rng.integers(0, n, size=n)
            X_b   = X.iloc[idx] if hasattr(X, 'iloc') else X[idx]
            y_b   = y.iloc[idx] if hasattr(y, 'iloc') else y[idx]
            if len(np.unique(y_b)) < 2:
                continue
            clf_b = LogisticRegression(
                penalty='l2', C=C,
                solver='lbfgs',
                max_iter=2000,
                random_state=42
            )
            try:
                clf_b.fit(X_b, y_b)
                boot_coefs.append(clf_b.coef_)
            except Exception:
                continue

        boot_coefs = np.array(boot_coefs)
        print(f"  Bootstrap: {len(boot_coefs)}/{n_bootstrap} iterations successful")

        rows = []
        for cls in clf.classes_:
            if cls == ref_class:
                continue
            cls_idx      = class_to_idx[cls]
            ref_idx_clf  = class_to_idx[ref_class]

            coef_diff      = clf.coef_[cls_idx] - clf.coef_[ref_idx_clf]
            boot_coef_diff = boot_coefs[:, cls_idx, :] - boot_coefs[:, ref_idx_clf, :]

            ci_low  = np.percentile(boot_coef_diff, 2.5,  axis=0)
            ci_high = np.percentile(boot_coef_diff, 97.5, axis=0)

            feat_cols = X.columns if hasattr(X, 'columns') else range(X.shape[1])
            for j, feat_col in enumerate(feat_cols):
                or_val = np.exp(coef_diff[j])
                ci_l   = np.exp(ci_low[j])
                ci_h   = np.exp(ci_high[j])
                sig    = not (ci_l <= 1.0 <= ci_h)
                clust_label = CLUSTER_LABELS.get(cls, f"C{cls}")
                ref_lbl     = CLUSTER_LABELS.get(ref_class, f"C{ref_class}")
                rows.append({
                    "cluster_vs_ref" : f"{clust_label} vs {ref_lbl}",
                    "variable"       : feat_col,
                    "OR"             : round(or_val, 3),
                    "CI_low"         : round(ci_l,   3),
                    "CI_high"        : round(ci_h,   3),
                    "significant"    : sig,
                    "model"          : "ridge_l2",
                })
        return pd.DataFrame(rows)

    # ── Forest plot helper ─────────────────────────────────────────────────────
    def draw_ridge_forest(df_results, clusters_vs_ref, suffix, out_dir, ref_label, C_val):
        for clust_label in clusters_vs_ref:
            df_plot = (
                df_results[df_results["cluster_vs_ref"] == clust_label]
                .sort_values("OR", ascending=True)
                .reset_index(drop=True)
            )
            n_vars = len(df_plot)
            colors = [
                "crimson"   if (s and or_val > 1)  else
                "steelblue" if (s and or_val <= 1) else
                "lightgrey"
                for s, or_val in zip(df_plot["significant"], df_plot["OR"])
            ]
            fig, ax = plt.subplots(figsize=(8, max(6, n_vars * 0.35)), facecolor="white")
            ax.set_facecolor("white")
            for i in range(n_vars):
                if i % 2 == 0:
                    ax.axhspan(i - 0.5, i + 0.5, color="lightgrey", alpha=0.3, zorder=0)
            for i in range(n_vars):
                ax.plot(
                    [df_plot["CI_low"].iloc[i], df_plot["CI_high"].iloc[i]],
                    [i, i], color=colors[i], linewidth=1.5, zorder=1
                )
                ax.scatter(df_plot["OR"].iloc[i], i, color=colors[i], s=40, zorder=2, marker="o")
            ax.axvline(1, color="black", linewidth=0.8, linestyle="-")
            for x in [0.5, 2.0]:
                ax.axvline(x, color="grey", linewidth=0.6, linestyle="--", alpha=0.6)
            ax.set_xscale("log")
            ax.set_yticks(range(n_vars))
            ax.set_yticklabels(df_plot["variable"], fontsize=9)
            ax.set_xlabel(f"Odds Ratio (log scale) — Ridge L2 bootstrap 95% CI (C={C_val})", fontsize=9)
            ax.set_title(
                f"{clust_label}\nref = {ref_label} | Ridge L2 (C={C_val}, 5-fold CV selected)",
                fontsize=10, fontweight="bold", color="black"
            )
            ax.spines["top"].set_visible(False)
            ax.spines["right"].set_visible(False)
            plt.tight_layout()
            fname_clean = (
                clust_label.replace(" ", "_").replace("—", "-")
                .replace("+", "plus").replace("/", "-")
                .replace("(", "").replace(")", "")
            )
            fname = f"forest_plot_ridge_{fname_clean}_{n_clusters}clusters_{suffix}.png"
            plt.savefig(os.path.join(out_dir, fname), dpi=200, bbox_inches="tight", facecolor="white")
            plt.close()
            print(f"Saved: {fname}")

        # ── Combined forest plot — all clusters ───────────────────────────────
        n_clusters_plot = len(clusters_vs_ref)
        n_cols_fp = min(3, n_clusters_plot)
        n_rows_fp = (n_clusters_plot + n_cols_fp - 1) // n_cols_fp

        fig, axes = plt.subplots(
            n_rows_fp, n_cols_fp,
            figsize=(n_cols_fp * 8, n_rows_fp * 6)
        )
        axes = np.array(axes).flatten()

        for ax_i, clust_label in enumerate(clusters_vs_ref):
            ax = axes[ax_i]
            df_plot = (
                df_results[df_results["cluster_vs_ref"] == clust_label]
                .sort_values("OR", ascending=True)
                .reset_index(drop=True)
            )
            n_vars_i = len(df_plot)
            colors_i = [
                "crimson"   if (s and or_val > 1)  else
                "steelblue" if (s and or_val <= 1) else
                "lightgrey"
                for s, or_val in zip(df_plot["significant"], df_plot["OR"])
            ]
            for i in range(n_vars_i):
                if i % 2 == 0:
                    ax.axhspan(i - 0.5, i + 0.5, color="lightgrey", alpha=0.3, zorder=0)
            for i in range(n_vars_i):
                ax.plot(
                    [df_plot["CI_low"].iloc[i], df_plot["CI_high"].iloc[i]],
                    [i, i], color=colors_i[i], linewidth=1.5, zorder=1
                )
                ax.scatter(df_plot["OR"].iloc[i], i, color=colors_i[i], s=30, zorder=2)
            ax.axvline(1, color="black", linewidth=0.8, linestyle="-")
            for x in [0.5, 2.0]:
                ax.axvline(x, color="grey", linewidth=0.6, linestyle="--", alpha=0.6)
            ax.set_xscale("log")
            ax.set_yticks(range(n_vars_i))
            ax.set_yticklabels(df_plot["variable"], fontsize=7)
            ax.set_xlabel("OR (log scale)", fontsize=9)
            ax.set_title(clust_label, fontsize=8, fontweight="bold")
            ax.spines["top"].set_visible(False)
            ax.spines["right"].set_visible(False)

        for j in range(ax_i + 1, len(axes)):
            axes[j].set_visible(False)

        plt.suptitle(
            f"Ridge L2 OR — all clusters vs {ref_label} — {n_clusters}clusters | C={C_val}\n"
            f"(crimson = sig OR>1 | steelblue = sig OR<1 | grey = ns | bootstrap 95% CI)",
            fontsize=12, fontweight="bold", y=1.01
        )
        plt.tight_layout()
        plt.savefig(
            os.path.join(out_dir, f"forest_plot_ridge_all_clusters_{n_clusters}clusters_{suffix}.png"),
            dpi=200, bbox_inches="tight"
        )
        plt.close()
        print(f"Saved: forest_plot_ridge_all_clusters_{n_clusters}clusters_{suffix}.png")

    # ══════════════════════════════════════════════════════════════════════════
    # WITH TRIAGE
    # ══════════════════════════════════════════════════════════════════════════
    print("\n── Encoding — with triage ──")
    X_wt, feat_map_wt = encode_with_ref(
        df_clusters_only, ALL_RIDGE_WITH_TRIAGE, REF_CATEGORIES
    )
    y_wt = df_clusters_only['cluster']
    print(f"  Features: {X_wt.shape[1]} dummies | N = {len(X_wt):,}")

    print("\n── Cross-validation — with triage ──")
    C_wt, C_wt_per_class = select_C_by_cv(X_wt, y_wt)

    print(f"\n── Ridge fit + bootstrap — with triage (C={C_wt}) ──")
    df_ridge_wt = ridge_results_with_ci(
        X_wt, y_wt,
        classes      = sorted(df_clusters_only['cluster'].unique()),
        ref_class    = ref_c,
        C            = C_wt,
        n_bootstrap  = 200,
    )
    df_ridge_wt['variable'] = df_ridge_wt['variable'].map(
        lambda x: feat_map_wt.get(x, x)
    )
    df_ridge_wt['C_optimal'] = C_wt
    df_ridge_wt.to_csv(
        os.path.join(OUT_DIR_REG, f"ridge_results_{n_clusters}clusters_with_triage.csv"),
        index=False
    )
    print(f"Saved: ridge_results_{n_clusters}clusters_with_triage.csv")

    clusters_vs_ref_wt = sorted(df_ridge_wt["cluster_vs_ref"].unique())
    draw_ridge_forest(df_ridge_wt, clusters_vs_ref_wt, "with_triage", OUT_DIR_REG, ref_label, C_wt)

    # ══════════════════════════════════════════════════════════════════════════
    # WITHOUT TRIAGE
    # ══════════════════════════════════════════════════════════════════════════
    print("\n── Encoding — without triage ──")
    X_nt, feat_map_nt = encode_with_ref(
        df_clusters_only, ALL_RIDGE_NO_TRIAGE, REF_CATEGORIES
    )
    y_nt = df_clusters_only['cluster']
    print(f"  Features: {X_nt.shape[1]} dummies | N = {len(X_nt):,}")

    print("\n── Cross-validation — without triage ──")
    C_nt, C_nt_per_class = select_C_by_cv(X_nt, y_nt)

    print(f"\n── Ridge fit + bootstrap — without triage (C={C_nt}) ──")
    df_ridge_nt = ridge_results_with_ci(
        X_nt, y_nt,
        classes      = sorted(df_clusters_only['cluster'].unique()),
        ref_class    = ref_c,
        C            = C_nt,
        n_bootstrap  = 200,
    )
    df_ridge_nt['variable'] = df_ridge_nt['variable'].map(
        lambda x: feat_map_nt.get(x, x)
    )
    df_ridge_nt['C_optimal'] = C_nt
    df_ridge_nt.to_csv(
        os.path.join(OUT_DIR_REG, f"ridge_results_{n_clusters}clusters_no_triage.csv"),
        index=False
    )
    print(f"Saved: ridge_results_{n_clusters}clusters_no_triage.csv")

    clusters_vs_ref_nt = sorted(df_ridge_nt["cluster_vs_ref"].unique())
    draw_ridge_forest(df_ridge_nt, clusters_vs_ref_nt, "no_triage", OUT_DIR_REG, ref_label, C_nt)

    print(f"\nCompleted Ridge — {n_clusters} clusters")
    print(f"  C with triage    : {C_wt}")
    print(f"  C without triage : {C_nt}")

In [302]:
# ==============================================================================
# CHUNK 5ter — Ridge L2 with DETAILED vital sign modalities
# ==============================================================================

from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.model_selection import StratifiedKFold
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt

# ── Reference categories for DETAILED vital signs ─────────────────────────────
REF_CATEGORIES_DETAILED = {
    "sex"                         : "M",
    "transport_grouped"           : "Personal",
    "age_group"                   : "15-30",
    "complaint_category_reg"      : "Trauma",
    "triage"              : "3",
    # Detailed vital sign references — clinical norm
    "bp_status"                   : "normotension",
    "hr_status"                   : "normocardia",
    "temp_status"                 : "normothermia",
    "sat_status"                  : "normal",
    "rr_status"                   : "normal",
    "o2_flow_status"              : "off",
    "gcs_status"                  : "normal",
    "cap_blood_sugar_status"      : "normoglycemia",
    "anisocoria_status"           : "no",
    "urine_dipstick_clean_status" : "negative",
    "pain_status"                 : "no_pain",
    "breathalyzer_status"         : "negative",
    "hemocue_status"              : "normal",
    "pupils_status"               : "normal",
}

# ── Feature lists for detailed Ridge ──────────────────────────────────────────
RIDGE_DETAILED_STATUS = [
    "bp_status", "hr_status", "temp_status", "sat_status",
    "rr_status", "o2_flow_status", "gcs_status",
    "cap_blood_sugar_status", "pain_status",
    # excluded due to excessive quasi-separation even with Ridge:
    "anisocoria_status", "urine_dipstick_clean_status",
    "breathalyzer_status", "hemocue_status", "pupils_status"
]

RIDGE_DETAILED_CATEG_WITH_TRIAGE = [
    "sex", "age_group", "complaint_category_reg",
    "triage", "transport_grouped"
]
RIDGE_DETAILED_CATEG_NO_TRIAGE = [
    "sex", "age_group", "complaint_category_reg", "transport_grouped"
]

ALL_RIDGE_DETAILED_WITH_TRIAGE = RIDGE_DETAILED_CATEG_WITH_TRIAGE + RIDGE_DETAILED_STATUS
ALL_RIDGE_DETAILED_NO_TRIAGE   = RIDGE_DETAILED_CATEG_NO_TRIAGE   + RIDGE_DETAILED_STATUS

for config in CLUSTERING_RUNS:
    n_clusters     = config['n_clusters']
    mcs            = config['mcs']
    ms             = config['ms']
    ref_c          = config['ref_cluster']
    CLUSTER_LABELS = config['cluster_labels']
    CLUSTER_ORDER  = config['cluster_order']
    OUT_DIR_REG    = f"{BASE_DIR}/final_mcs{mcs}_ms{ms}/logistic_regression"
    os.makedirs(OUT_DIR_REG, exist_ok=True)

    # ── Load — NO vital sign remap (keep original modalities) ─────────────────
    CSV_PATH = f"{BASE_DIR}/final_mcs{mcs}_ms{ms}/clustering_{RUN_LABEL}_{SCALER}_mcs{mcs}_ms{ms}_labeled.csv"
    df_clust = pd.read_csv(CSV_PATH, low_memory=False)

    # Triage + complaint only — NO vital sign remap
    # df_clust['triage_grouped'] = (
    #     df_clust['triage'].astype(float).astype(int)
    #     .map(TRIAGE_GROUP_MAP).astype(str)
    # )
    df_clust['triage'] = df_clust['triage'].astype(float).astype(int).astype(str)
    df_clust['complaint_category_reg'] = df_clust['complaint_category'].replace(
        {cat: 'Other' for cat in SYSTEMIC_RARE}
    )

    # ── Build df_reg ──────────────────────────────────────────────────────────
    cols_needed = list(set(ALL_RIDGE_DETAILED_WITH_TRIAGE + ['cluster']))
    df_reg = df_clust[[c for c in cols_needed if c in df_clust.columns]].copy()

    # Convert to string to avoid NaN from unmapped values
    for col in ALL_RIDGE_DETAILED_WITH_TRIAGE:
        if col in df_reg.columns:
            df_reg[col] = df_reg[col].astype(str).replace('nan', 'not_measured')

    df_reg = df_reg.dropna(subset=ALL_RIDGE_DETAILED_WITH_TRIAGE + ['cluster'])
    df_reg['cluster'] = df_reg['cluster'].astype(int)
    df_clusters_only  = df_reg[df_reg['cluster'] != -1].copy()

    ref_label = CLUSTER_LABELS.get(ref_c, f"C{ref_c}")

    print("\n" + "="*60)
    print(f"RIDGE L2 DETAILED MODALITIES — {n_clusters} clusters")
    print("="*60)
    print(f"N = {len(df_clusters_only):,} patients")

    # ── Helper — one-hot encode with explicit reference ────────────────────────
    def encode_with_ref_detailed(df, features, ref_categories):
        dummies_list = []
        feature_map  = {}
        for feat in features:
            if feat not in df.columns:
                print(f"  WARNING: {feat} not in df")
                continue
            ref     = ref_categories.get(feat, df[feat].mode()[0])
            dummies = pd.get_dummies(df[feat], prefix=feat, drop_first=False)
            ref_col = f"{feat}_{ref}"
            if ref_col not in dummies.columns:
                print(f"  WARNING: reference '{ref}' not found for {feat}")
                print(f"    Available: {[c.replace(feat+'_','') for c in dummies.columns]}")
            else:
                dummies = dummies.drop(columns=[ref_col])
            for col in dummies.columns:
                modality = col.replace(f"{feat}_", "")
                feature_map[col] = f"{feat} = {modality}"
            dummies_list.append(dummies)
        X = pd.concat(dummies_list, axis=1).astype(float)
        return X, feature_map

    # ──  CV selection of C ─────────────────────────────────────────────
    def select_C_by_cv(
        X,
        y,
        stratify_var=None,
        Cs=None,
        n_splits=5,
        random_state=42
    ):

        # Dense logarithmic grid (glmnet-like)
        if Cs is None:
            Cs = np.logspace(-3, 2, 50)

        # Joint stratification:
        # preserve both cluster distribution and triage distribution
        if stratify_var is not None:
            stratify_labels = (
                y.astype(str) + "_" + stratify_var.astype(str)
            )
        else:
            stratify_labels = y.astype(str)

        cv_base = StratifiedKFold(
            n_splits=n_splits,
            shuffle=True,
            random_state=random_state
        )

        cv_splits = list(cv_base.split(X, stratify_labels))

        clf_cv = LogisticRegressionCV(
            Cs           = Cs,
            cv           = cv_splits,
            penalty      = 'l2',
            solver       = 'lbfgs',
            max_iter     = 2000,
            scoring      = 'neg_log_loss',
            random_state = random_state,
            n_jobs       = 128,
        )

        clf_cv.fit(X, y)

        C_optimal = float(np.mean(clf_cv.C_))

        print(f"  C per class : {clf_cv.C_}")
        print(f"  C optimal   : {C_optimal}")

        return C_optimal

    # ──  Ridge + bootstrap CI ─────────────────────────────────────────
    def ridge_results_with_ci(X, y, classes, ref_class, C=1.0, n_bootstrap=200, random_state=42):
        rng = np.random.default_rng(random_state)
        n   = len(X)

        clf = LogisticRegression(
            penalty='l2', C=C, solver='lbfgs', max_iter=2000, random_state=42
        )
        clf.fit(X, y)
        class_to_idx = {c: i for i, c in enumerate(clf.classes_)}

        boot_coefs = []
        for _ in range(n_bootstrap):
            idx  = rng.integers(0, n, size=n)
            X_b  = X.iloc[idx]
            y_b  = y.iloc[idx]
            if len(np.unique(y_b)) < 2:
                continue
            clf_b = LogisticRegression(
                penalty='l2', C=C, solver='lbfgs', max_iter=2000, random_state=42
            )
            try:
                clf_b.fit(X_b, y_b)
                boot_coefs.append(clf_b.coef_)
            except Exception:
                continue

        boot_coefs = np.array(boot_coefs)
        print(f"  Bootstrap: {len(boot_coefs)}/{n_bootstrap} iterations successful")

        rows = []
        for cls in clf.classes_:
            if cls == ref_class:
                continue
            cls_idx     = class_to_idx[cls]
            ref_idx_clf = class_to_idx[ref_class]

            coef_diff      = clf.coef_[cls_idx] - clf.coef_[ref_idx_clf]
            boot_coef_diff = boot_coefs[:, cls_idx, :] - boot_coefs[:, ref_idx_clf, :]
            ci_low  = np.percentile(boot_coef_diff, 2.5,  axis=0)
            ci_high = np.percentile(boot_coef_diff, 97.5, axis=0)

            for j, feat_col in enumerate(X.columns):
                or_val = np.exp(coef_diff[j])
                ci_l   = np.exp(ci_low[j])
                ci_h   = np.exp(ci_high[j])
                sig    = not (ci_l <= 1.0 <= ci_h)
                rows.append({
                    "cluster_vs_ref" : f"{CLUSTER_LABELS.get(cls, f'C{cls}')} vs {CLUSTER_LABELS.get(ref_class, f'C{ref_class}')}",
                    "variable"       : feat_col,
                    "OR"             : round(or_val, 3),
                    "CI_low"         : round(ci_l,   3),
                    "CI_high"        : round(ci_h,   3),
                    "significant"    : sig,
                    "model"          : "ridge_l2_detailed",
                })
        return pd.DataFrame(rows)

    # ── forest plot ───────────────────────────────────────────────────
    def draw_ridge_forest_detailed(df_results, clusters_vs_ref, suffix, out_dir, ref_label, C_val):
        # Individual plots
        for clust_label in clusters_vs_ref:
            df_plot = (
                df_results[df_results["cluster_vs_ref"] == clust_label]
                .sort_values("OR", ascending=True)
                .reset_index(drop=True)
            )
            n_vars = len(df_plot)
            colors = [
                "crimson"   if (s and or_val > 1)  else
                "steelblue" if (s and or_val <= 1) else
                "lightgrey"
                for s, or_val in zip(df_plot["significant"], df_plot["OR"])
            ]
            fig, ax = plt.subplots(figsize=(8, max(6, n_vars * 0.35)), facecolor="white")
            ax.set_facecolor("white")
            for i in range(n_vars):
                if i % 2 == 0:
                    ax.axhspan(i - 0.5, i + 0.5, color="lightgrey", alpha=0.3, zorder=0)
            for i in range(n_vars):
                ax.plot(
                    [df_plot["CI_low"].iloc[i], df_plot["CI_high"].iloc[i]],
                    [i, i], color=colors[i], linewidth=1.5, zorder=1
                )
                ax.scatter(df_plot["OR"].iloc[i], i, color=colors[i], s=40, zorder=2)
            ax.axvline(1, color="black", linewidth=0.8, linestyle="-")
            for x in [0.5, 2.0]:
                ax.axvline(x, color="grey", linewidth=0.6, linestyle="--", alpha=0.6)
            ax.set_xscale("log")
            ax.set_yticks(range(n_vars))
            ax.set_yticklabels(df_plot["variable"], fontsize=9)
            ax.set_xlabel(f"OR (log scale) — Ridge L2 detailed, bootstrap 95% CI (C={C_val})", fontsize=9)
            ax.set_title(
                f"{clust_label}\nref = {ref_label} | Ridge L2 detailed modalities (C={C_val})",
                fontsize=10, fontweight="bold", color="black"
            )
            ax.spines["top"].set_visible(False)
            ax.spines["right"].set_visible(False)
            plt.tight_layout()
            fname_clean = (
                clust_label.replace(" ", "_").replace("—", "-")
                .replace("+", "plus").replace("/", "-")
                .replace("(", "").replace(")", "")
            )
            fname = f"forest_plot_ridge_detailed_{fname_clean}_{n_clusters}clusters_{suffix}.png"
            plt.savefig(os.path.join(out_dir, fname), dpi=200, bbox_inches="tight", facecolor="white")
            plt.close()
            print(f"Saved: {fname}")

        # Combined plot
        n_clusters_plot = len(clusters_vs_ref)
        n_cols_fp = min(3, n_clusters_plot)
        n_rows_fp = (n_clusters_plot + n_cols_fp - 1) // n_cols_fp
        fig, axes = plt.subplots(
            n_rows_fp, n_cols_fp,
            figsize=(n_cols_fp * 8, n_rows_fp * 6)
        )
        axes = np.array(axes).flatten()
        for ax_i, clust_label in enumerate(clusters_vs_ref):
            ax = axes[ax_i]
            df_plot = (
                df_results[df_results["cluster_vs_ref"] == clust_label]
                .sort_values("OR", ascending=True)
                .reset_index(drop=True)
            )
            n_vars_i = len(df_plot)
            colors_i = [
                "crimson"   if (s and or_val > 1)  else
                "steelblue" if (s and or_val <= 1) else
                "lightgrey"
                for s, or_val in zip(df_plot["significant"], df_plot["OR"])
            ]
            for i in range(n_vars_i):
                if i % 2 == 0:
                    ax.axhspan(i - 0.5, i + 0.5, color="lightgrey", alpha=0.3, zorder=0)
            for i in range(n_vars_i):
                ax.plot(
                    [df_plot["CI_low"].iloc[i], df_plot["CI_high"].iloc[i]],
                    [i, i], color=colors_i[i], linewidth=1.5, zorder=1
                )
                ax.scatter(df_plot["OR"].iloc[i], i, color=colors_i[i], s=30, zorder=2)
            ax.axvline(1, color="black", linewidth=0.8, linestyle="-")
            for x in [0.5, 2.0]:
                ax.axvline(x, color="grey", linewidth=0.6, linestyle="--", alpha=0.6)
            ax.set_xscale("log")
            ax.set_yticks(range(n_vars_i))
            ax.set_yticklabels(df_plot["variable"], fontsize=7)
            ax.set_xlabel("OR (log scale)", fontsize=9)
            ax.set_title(clust_label, fontsize=8, fontweight="bold")
            ax.spines["top"].set_visible(False)
            ax.spines["right"].set_visible(False)
        for j in range(ax_i + 1, len(axes)):
            axes[j].set_visible(False)
        plt.suptitle(
            f"Ridge L2 detailed — all clusters vs {ref_label} — {n_clusters} clusters | C={C_val}\n"
            f"(crimson = sig OR>1 | steelblue = sig OR<1 | grey = ns | bootstrap 95% CI)",
            fontsize=12, fontweight="bold", y=1.01
        )
        plt.tight_layout()
        plt.savefig(
            os.path.join(out_dir, f"forest_plot_ridge_detailed_all_clusters_{n_clusters}clusters_{suffix}.png"),
            dpi=200, bbox_inches="tight"
        )
        plt.close()
        print(f"Saved: forest_plot_ridge_detailed_all_clusters_{n_clusters}clusters_{suffix}.png")

    # ══════════════════════════════════════════════════════════════════════════
    # WITH TRIAGE
    # ══════════════════════════════════════════════════════════════════════════
    print("\n── Encoding — with triage (detailed) ──")
    X_wt, feat_map_wt = encode_with_ref_detailed(
        df_clusters_only, ALL_RIDGE_DETAILED_WITH_TRIAGE, REF_CATEGORIES_DETAILED
    )
    y_wt = df_clusters_only['cluster']
    print(f"  Features: {X_wt.shape[1]} dummies | N = {len(X_wt):,}")

    print("\n── Cross-validation — with triage (detailed) ──")
    C_wt = select_C_by_cv(X_wt, y_wt)

    print(f"\n── Ridge fit + bootstrap — with triage detailed (C={C_wt}) ──")
    df_ridge_wt = ridge_results_with_ci(
        X_wt, y_wt,
        classes     = sorted(df_clusters_only['cluster'].unique()),
        ref_class   = ref_c,
        C           = C_wt,
        n_bootstrap = 200,
    )
    df_ridge_wt['variable']  = df_ridge_wt['variable'].map(lambda x: feat_map_wt.get(x, x))
    df_ridge_wt['C_optimal'] = C_wt
    # ── Flag unstable estimates ────────────────────────────────────────────────────
    df_ridge_wt['CI_ratio']  = df_ridge_wt['CI_high'] / df_ridge_wt['CI_low']
    df_ridge_wt['unstable']  = df_ridge_wt['CI_ratio'] > 20

    n_unstable = df_ridge_wt['unstable'].sum()
    if n_unstable > 0:
        print(f"WARNING — {n_unstable} unstable estimates (CI_ratio > 20):")
        print(df_ridge_wt[df_ridge_wt['unstable']][
            ['cluster_vs_ref', 'variable', 'OR', 'CI_low', 'CI_high', 'CI_ratio']
        ].to_string())
    df_ridge_wt.to_csv(
        os.path.join(OUT_DIR_REG, f"ridge_detailed_results_{n_clusters}clusters_with_triage.csv"),
        index=False
    )
    print(f"Saved: ridge_detailed_results_{n_clusters}clusters_with_triage.csv")
    clusters_vs_ref_wt = sorted(df_ridge_wt["cluster_vs_ref"].unique())
    draw_ridge_forest_detailed(df_ridge_wt, clusters_vs_ref_wt, "with_triage", OUT_DIR_REG, ref_label, C_wt)

    # ══════════════════════════════════════════════════════════════════════════
    # WITHOUT TRIAGE
    # ══════════════════════════════════════════════════════════════════════════
    print("\n── Encoding — without triage (detailed) ──")
    X_nt, feat_map_nt = encode_with_ref_detailed(
        df_clusters_only, ALL_RIDGE_DETAILED_NO_TRIAGE, REF_CATEGORIES_DETAILED
    )
    y_nt = df_clusters_only['cluster']
    print(f"  Features: {X_nt.shape[1]} dummies | N = {len(X_nt):,}")

    print("\n── Cross-validation — without triage (detailed) ──")
    C_nt = select_C_by_cv(X_nt, y_nt)

    print(f"\n── Ridge fit + bootstrap — without triage detailed (C={C_nt}) ──")
    df_ridge_nt = ridge_results_with_ci(
        X_nt, y_nt,
        classes     = sorted(df_clusters_only['cluster'].unique()),
        ref_class   = ref_c,
        C           = C_nt,
        n_bootstrap = 200,
    )
    df_ridge_nt['variable']  = df_ridge_nt['variable'].map(lambda x: feat_map_nt.get(x, x))
    df_ridge_nt['C_optimal'] = C_nt
    # ── Flag unstable estimates ────────────────────────────────────────────────────
    df_ridge_nt['CI_ratio']  = df_ridge_nt['CI_high'] / df_ridge_nt['CI_low']
    df_ridge_nt['unstable']  = df_ridge_nt['CI_ratio'] > 20

    n_unstable_nt = df_ridge_nt['unstable'].sum()
    if n_unstable_nt > 0:
        print(f"WARNING — {n_unstable_nt} unstable estimates (CI_ratio > 20):")
        print(df_ridge_nt[df_ridge_nt['unstable']][
            ['cluster_vs_ref', 'variable', 'OR', 'CI_low', 'CI_high', 'CI_ratio']
        ].to_string())

    df_ridge_nt.to_csv(
        os.path.join(OUT_DIR_REG, f"ridge_detailed_results_{n_clusters}clusters_no_triage.csv"),
        index=False
    )
    print(f"Saved: ridge_detailed_results_{n_clusters}clusters_no_triage.csv")
    clusters_vs_ref_nt = sorted(df_ridge_nt["cluster_vs_ref"].unique())
    draw_ridge_forest_detailed(df_ridge_nt, clusters_vs_ref_nt, "no_triage", OUT_DIR_REG, ref_label, C_nt)

    print(f"\nCompleted Ridge detailed — {n_clusters} clusters")
    print(f"  C with triage    : {C_wt}")
    print(f"  C without triage : {C_nt}")





RIDGE L2 DETAILED MODALITIES — 5 clusters
N = 54,886 patients

── Encoding — with triage (detailed) ──
  Features: 62 dummies | N = 54,886

── Cross-validation — with triage (detailed) ──
  C per class : [12.06792641 12.06792641 12.06792641 12.06792641 12.06792641]
  C optimal   : 12.067926406393289

── Ridge fit + bootstrap — with triage detailed (C=12.067926406393289) ──
  Bootstrap: 200/200 iterations successful
Saved: ridge_detailed_results_5clusters_with_triage.csv
Saved: forest_plot_ridge_detailed_C1_-_UHCD_plus_hospitalization_plus_heavy_workup_vs_C5_-_Discharged_plus_minimal_consumption_5clusters_with_triage.png
Saved: forest_plot_ridge_detailed_C2_-_Hospitalized_plus_full_workup_vs_C5_-_Discharged_plus_minimal_consumption_5clusters_with_triage.png
Saved: forest_plot_ridge_detailed_C3_-_Discharged_plus_biology_plus--_ECG_vs_C5_-_Discharged_plus_minimal_consumption_5clusters_with_triage.png
Saved: forest_plot_ridge_detailed_C4_-_Discharged_plus_isolated_X-ray_plus--_CT_vs_C5_-_

In [309]:
import sklearn
print(sklearn.__version__)

1.8.0


In [303]:
df = pd.read_csv('Results/Regular_clustering/Full_dataset/With_counts/minmax/s2_balanced/final_mcs2271_ms15/logistic_regression/ridge_detailed_results_9clusters_with_triage.csv')
print(df[df['OR'] < 0.02][['cluster_vs_ref','variable','OR','CI_low','CI_high']])


                                        cluster_vs_ref  \
132  C8 — Discharged + isolated X-ray vs C9 — Disch...   

                                              variable     OR  CI_low  CI_high  
132  complaint_category_reg = ENT_Ophthalmology_Dental  0.005   0.003    0.008  


In [305]:
# ==============================================================================
# CHUNK 5ter_bis — Model comparison Ridge detailed (with vs without triage)
# ==============================================================================

from sklearn.metrics import log_loss
from scipy.stats import chi2
import numpy as np
import pandas as pd
import os

# Rebuild models from scratch using stored X, y and C values
# Requires: X_wt, X_nt, y_wt, y_nt, C_wt, C_nt, n_clusters, OUT_DIR_REG
# These are in memory from the last iteration of CLUSTERING_RUNS in chunk 5ter

from sklearn.linear_model import LogisticRegression

clf_wt_comp = LogisticRegression(
    penalty='l2', C=C_wt, solver='lbfgs', max_iter=2000, random_state=42
)
clf_wt_comp.fit(X_wt, y_wt)

clf_nt_comp = LogisticRegression(
    penalty='l2', C=C_nt, solver='lbfgs', max_iter=2000, random_state=42
)
clf_nt_comp.fit(X_nt, y_nt)

# Null model
y_arr = y_wt.values
class_proportions = np.bincount(
    np.searchsorted(np.unique(y_arr), y_arr)
) / len(y_arr)

ll_null = -log_loss(y_wt, np.tile(class_proportions, (len(y_wt), 1)),
                    normalize=False)
ll_wt   = -log_loss(y_wt, clf_wt_comp.predict_proba(X_wt), normalize=False)
ll_nt   = -log_loss(y_nt, clf_nt_comp.predict_proba(X_nt), normalize=False)

r2_wt   = 1 - (ll_wt / ll_null)
r2_nt   = 1 - (ll_nt / ll_null)
lr_stat = 2 * (ll_wt - ll_nt)
df_diff = X_wt.shape[1] - X_nt.shape[1]
p_lrt   = chi2.sf(lr_stat, df_diff)

print(f"── Model comparison — {n_clusters} clusters ──")
print(f"  McFadden R² with triage    : {r2_wt:.4f}")
print(f"  McFadden R² without triage : {r2_nt:.4f}")
print(f"  Delta R²                   : {r2_wt - r2_nt:.4f}")
print(f"  LRT statistic (approx)     : {lr_stat:.2f}")
print(f"  df difference              : {df_diff}")
print(f"  p-value                    : {p_lrt:.4e}")
print(f"  NOTE: approximate — Ridge log-loss != MLE log-likelihood")

pd.DataFrame([{
    'n_clusters'      : n_clusters,
    'McFadden_R2_wt'  : round(r2_wt,  4),
    'McFadden_R2_nt'  : round(r2_nt,  4),
    'delta_R2'        : round(r2_wt - r2_nt, 4),
    'LRT_stat_approx' : round(lr_stat, 2),
    'df_diff'         : df_diff,
    'p_lrt_approx'    : round(p_lrt,  6),
    'C_wt'            : C_wt,
    'C_nt'            : C_nt,
}]).to_csv(
    os.path.join(OUT_DIR_REG,
        f"ridge_detailed_model_comparison_{n_clusters}clusters.csv"),
    index=False
)
print(f"Saved: ridge_detailed_model_comparison_{n_clusters}clusters.csv")

── Model comparison — 9 clusters ──
  McFadden R² with triage    : 0.3115
  McFadden R² without triage : 0.2849
  Delta R²                   : 0.0266
  LRT statistic (approx)     : 5685.53
  df difference              : 4
  p-value                    : 0.0000e+00
  NOTE: approximate — Ridge log-loss != MLE log-likelihood
Saved: ridge_detailed_model_comparison_9clusters.csv


In [308]:
# ==============================================================================
# CHUNK 5quater — Binary Ridge L2 logistic regression: Outliers vs all clusters
# ==============================================================================

from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.model_selection import StratifiedKFold
import statsmodels.formula.api as smf
import pandas as pd
import numpy as np
import os
import re
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

for config in CLUSTERING_RUNS:
    n_clusters     = config['n_clusters']
    mcs            = config['mcs']
    ms             = config['ms']
    ref_c          = config['ref_cluster']
    CLUSTER_LABELS = config['cluster_labels']
    CLUSTER_ORDER  = config['cluster_order']
    OUT_DIR_REG    = f"{BASE_DIR}/final_mcs{mcs}_ms{ms}/logistic_regression"
    os.makedirs(OUT_DIR_REG, exist_ok=True)

    # ── Load — NO vital sign remap ────────────────────────────────────────────
    CSV_PATH = f"{BASE_DIR}/final_mcs{mcs}_ms{ms}/clustering_{RUN_LABEL}_{SCALER}_mcs{mcs}_ms{ms}_labeled.csv"
    df_clust = pd.read_csv(CSV_PATH, low_memory=False)

    df_clust['triage'] = df_clust['triage'].astype(float).astype(int).astype(str)
    df_clust['complaint_category_reg'] = df_clust['complaint_category'].replace(
        {cat: 'Other' for cat in SYSTEMIC_RARE}
    )

    # ── Build df_reg (ALL patients including outliers) ────────────────────────
    BINARY_FEATURES = RIDGE_DETAILED_CATEG_WITH_TRIAGE + RIDGE_DETAILED_STATUS
    cols_needed = list(set(BINARY_FEATURES + ['cluster']))
    df_reg = df_clust[[c for c in cols_needed if c in df_clust.columns]].copy()

    for col in BINARY_FEATURES:
        if col in df_reg.columns:
            df_reg[col] = df_reg[col].astype(str).replace('nan', 'not_measured')

    df_reg = df_reg.dropna(subset=BINARY_FEATURES + ['cluster'])
    df_reg['cluster']    = df_reg['cluster'].astype(int)
    df_reg['is_outlier'] = (df_reg['cluster'] == -1).astype(int)

    print("\n" + "="*60)
    print(f"BINARY RIDGE L2 — OUTLIERS vs ALL — {n_clusters} clusters")
    print("="*60)
    print(f"N total  : {len(df_reg):,}")
    print(f"Outliers : {df_reg['is_outlier'].sum():,} ({df_reg['is_outlier'].mean():.1%})")

    # ── Encode ────────────────────────────────────────────────────────────────
    X_bin, feat_map_bin = encode_with_ref_detailed(
        df_reg, BINARY_FEATURES, REF_CATEGORIES_DETAILED
    )
    y_bin = df_reg['is_outlier']
    print(f"Features: {X_bin.shape[1]} dummies | N = {len(X_bin):,}")

    # ── CV selection of C ─────────────────────────────────────────────────────
    print("\n── Cross-validation ──")

    # Dense logarithmic grid (glmnet-like)
    Cs_grid = np.logspace(-3, 2, 50)

    # Stratification preserving:
    # - outlier proportion
    # - triage score distribution
    stratify_labels = (
        y_bin.astype(str) + "_" + df_reg["triage"].astype(str)
    )

    cv_base = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    )

    cv_splits = list(cv_base.split(X_bin, stratify_labels))

    clf_cv = LogisticRegressionCV(
        Cs           = Cs_grid,
        cv           = cv_splits,
        penalty      = 'l2',
        solver       = 'lbfgs',
        max_iter     = 2000,
        scoring      = 'neg_log_loss',
        random_state = 42,
        n_jobs       = 128,
    )

    clf_cv.fit(X_bin, y_bin)

    C_bin = float(clf_cv.C_[0])

    print(f"  C optimal : {C_bin}")

    # ── Ridge fit ─────────────────────────────────────────────────────────────
    print(f"\n── Ridge fit + bootstrap (C={C_bin}) ──")
    clf_bin = LogisticRegression(
        penalty='l2', C=C_bin, solver='lbfgs', max_iter=2000, random_state=42
    )
    clf_bin.fit(X_bin, y_bin)

    # ── Bootstrap CI ──────────────────────────────────────────────────────────
    rng  = np.random.default_rng(42)
    n    = len(X_bin)
    boot_coefs = []

    for _ in range(200):
        idx = rng.integers(0, n, size=n)
        X_b = X_bin.iloc[idx]
        y_b = y_bin.iloc[idx]
        if y_b.nunique() < 2:
            continue
        clf_b = LogisticRegression(
            penalty='l2', C=C_bin, solver='lbfgs', max_iter=2000, random_state=42
        )
        try:
            clf_b.fit(X_b, y_b)
            boot_coefs.append(clf_b.coef_[0])
        except Exception:
            continue

    boot_coefs = np.array(boot_coefs)
    print(f"  Bootstrap: {len(boot_coefs)}/200 iterations successful")

    ci_low  = np.percentile(boot_coefs, 2.5,  axis=0)
    ci_high = np.percentile(boot_coefs, 97.5, axis=0)
    coef    = clf_bin.coef_[0]

    # ── Extract results ────────────────────────────────────────────────────────
    rows = []
    for j, feat_col in enumerate(X_bin.columns):
        or_val = np.exp(coef[j])
        ci_l   = np.exp(ci_low[j])
        ci_h   = np.exp(ci_high[j])
        sig    = not (ci_l <= 1.0 <= ci_h)
        rows.append({
            "variable"   : feat_map_bin.get(feat_col, feat_col),
            "OR"         : round(or_val, 3),
            "CI_low"     : round(ci_l,   3),
            "CI_high"    : round(ci_h,   3),
            "significant": sig,
            "model"      : "ridge_l2_binary",
            "C_optimal"  : C_bin,
        })

    df_binary_results = pd.DataFrame(rows)

    # ── Flag unstable ─────────────────────────────────────────────────────────
    df_binary_results['CI_ratio'] = df_binary_results['CI_high'] / df_binary_results['CI_low']
    df_binary_results['unstable'] = df_binary_results['CI_ratio'] > 20

    n_unstable = df_binary_results['unstable'].sum()
    if n_unstable > 0:
        print(f"WARNING — {n_unstable} unstable estimates (CI_ratio > 20):")
        print(df_binary_results[df_binary_results['unstable']][
            ['variable', 'OR', 'CI_low', 'CI_high', 'CI_ratio']
        ].to_string())

    df_binary_results.to_csv(
        os.path.join(OUT_DIR_REG,
            f"binary_outlier_ridge_results_{n_clusters}clusters.csv"),
        index=False
    )
    print(f"Saved: binary_outlier_ridge_results_{n_clusters}clusters.csv")

    # ── Forest plot ───────────────────────────────────────────────────────────
    df_plot = df_binary_results.sort_values("OR", ascending=True).reset_index(drop=True)
    n_vars  = len(df_plot)

    ci_low_disp  = df_plot.apply(
        lambda r: r["OR"] * 0.5 if r["unstable"] else r["CI_low"], axis=1
    ).values
    ci_high_disp = df_plot.apply(
        lambda r: r["OR"] * 2.0 if r["unstable"] else r["CI_high"], axis=1
    ).values
    or_vals = df_plot["OR"].values

    colors = [
        "crimson"   if (s and or_val > 1)  else
        "steelblue" if (s and or_val <= 1) else
        "lightgrey"
        for s, or_val in zip(df_plot["significant"], df_plot["OR"])
    ]

    fig, ax = plt.subplots(figsize=(8, max(6, n_vars * 0.35)), facecolor="white")
    ax.set_facecolor("white")

    for i in range(n_vars):
        if i % 2 == 0:
            ax.axhspan(i - 0.5, i + 0.5, color="lightgrey", alpha=0.3, zorder=0)

    for i in range(n_vars):
        ax.plot(
            [ci_low_disp[i], ci_high_disp[i]],
            [i, i],
            color=colors[i], linewidth=1.5, zorder=1,
            linestyle="--" if df_plot["unstable"].iloc[i] else "-"
        )
        marker = "^" if df_plot["unstable"].iloc[i] else "o"
        ax.scatter(or_vals[i], i, color=colors[i], s=40, zorder=2, marker=marker)

    ax.axvline(1, color="black", linewidth=0.8, linestyle="-")
    for x in [0.5, 2.0]:
        ax.axvline(x, color="grey", linewidth=0.6, linestyle="--", alpha=0.6)

    ax.set_xscale("log")
    ax.set_xticks([0.1, 0.25, 0.5, 1.0, 2.0, 4.0, 10.0])           # ← ajoute
    ax.get_xaxis().set_major_formatter(ticker.ScalarFormatter())      # ← ajoute
    ax.tick_params(axis='x', colors='black')
    for label in ax.get_xticklabels():
        label.set_color('black')
    ax.set_yticks(range(n_vars))
    ax.set_yticklabels(df_plot["variable"], fontsize=9, color='black')


    unstable_note = "\n(triangle = unstable estimate, CI_ratio > 20)" if n_unstable > 0 else ""
    ax.set_title(
        f"Binary Ridge L2 — Outliers vs all clusters — {n_clusters} clusters\n"
        f"(crimson = sig OR>1 | steelblue = sig OR<1 | grey = ns){unstable_note}",
        fontsize=10, fontweight="bold", color="black"
    )
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    plt.tight_layout()
    plt.savefig(
        os.path.join(OUT_DIR_REG,
            f"forest_plot_binary_outlier_ridge_{n_clusters}clusters.png"),
        dpi=200, bbox_inches="tight", facecolor="white"
    )
    plt.close()
    print(f"Saved: forest_plot_binary_outlier_ridge_{n_clusters}clusters.png")

    print(f"\nCompleted binary Ridge outlier regression — {n_clusters} clusters")
    print(f"  C optimal : {C_bin}")
    print(f"  N unstable: {n_unstable}")


BINARY RIDGE L2 — OUTLIERS vs ALL — 5 clusters
N total  : 56,784
Outliers : 1,898 (3.3%)
Features: 62 dummies | N = 56,784

── Cross-validation ──
  C optimal : 0.13894954943731375

── Ridge fit + bootstrap (C=0.13894954943731375) ──
  Bootstrap: 200/200 iterations successful
Saved: binary_outlier_ridge_results_5clusters.csv
Saved: forest_plot_binary_outlier_ridge_5clusters.png

Completed binary Ridge outlier regression — 5 clusters
  C optimal : 0.13894954943731375
  N unstable: 0

BINARY RIDGE L2 — OUTLIERS vs ALL — 9 clusters
N total  : 56,784
Outliers : 4,695 (8.3%)
Features: 62 dummies | N = 56,784

── Cross-validation ──
  C optimal : 0.9102981779915218

── Ridge fit + bootstrap (C=0.9102981779915218) ──
  Bootstrap: 200/200 iterations successful
Saved: binary_outlier_ridge_results_9clusters.csv
Saved: forest_plot_binary_outlier_ridge_9clusters.png

Completed binary Ridge outlier regression — 9 clusters
  C optimal : 0.9102981779915218
  N unstable: 0
